In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os

data_dir = "/kaggle/input/datasets/rautranjit/thai-dataset"

train_dir = os.path.join(data_dir, "train")
test_dir  = os.path.join(data_dir, "test")

train_classes = sorted([c for c in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, c))])
test_classes  = sorted([c for c in os.listdir(test_dir)  if os.path.isdir(os.path.join(test_dir,  c))])

print(f"Train classes : {len(train_classes)}")
print(f"Test  classes : {len(test_classes)}")
print(f"Match         : {set(train_classes) == set(test_classes)}")

# Show any mismatches
only_in_train = set(train_classes) - set(test_classes)
only_in_test  = set(test_classes)  - set(train_classes)
if only_in_train:
    print(f"Only in train : {sorted(only_in_train)}")
if only_in_test:
    print(f"Only in test  : {sorted(only_in_test)}")

print(f"\nFirst 10 train: {train_classes[:10]}")
print(f"First 10 test : {test_classes[:10]}")

Train classes : 68
Test  classes : 68
Match         : True

First 10 train: ['00-161-A1-KO KAI', '01-162-A2-KHO KHAI', '02-163-A3-KHO KHUAT', '03-164-A4-KHO KHWAI', '04-165-A5-KHO KHON', '05-166-A6-KHO RAKHANG', '06-167-A7-NGO NGU', '07-168-A8-CHO CHAN', '08-169-A9-CHO CHING', '09-170-AA-CHO CHANG']
First 10 test : ['00-161-A1-KO KAI', '01-162-A2-KHO KHAI', '02-163-A3-KHO KHUAT', '03-164-A4-KHO KHWAI', '04-165-A5-KHO KHON', '05-166-A6-KHO RAKHANG', '06-167-A7-NGO NGU', '07-168-A8-CHO CHAN', '08-169-A9-CHO CHING', '09-170-AA-CHO CHANG']


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  WhatNet  –  Thai Script Recognition
#  Adapted from the Kuzushiji version.
#
#  Transfer rationale
#  ──────────────────
#  Thai script is an abugida with distinctive circular loops at the top of many
#  consonants (e.g., ก, ข, ค).  Burapha-TH covers 44 consonants plus digit
#  classes.  Key structural differences vs. Kuzushiji:
#
#    • num_classes   : 49  → 54   (44 consonants + 10 digits)
#    • image_size    : 28  → 64   (Burapha-TH is natively 64×64)
#    • Scaffold bypass: REMOVED – Thai glyphs have complex embedded loops that
#                        a simple bypass convolution would smear rather than
#                        disentangle.  The full encoder is used end-to-end.
#    • Stem kernels  : 3×3 isotropic → 3×3 + 1×5 + 5×1
#                       • 3×3 standard  – general stroke body
#                       • 1×5 horizontal – vowel diacritics sit above/below
#                         the consonant baseline as thin horizontal strokes
#                       • 5×1 vertical   – vertical consonant stems and
#                         descenders common in Thai consonants
#    • STM kernels   : 3×3+3×1+dil(1,3) → circular 5×5 + dil(1,2)
#                       • 5×5 standard  – captures the full circular loop
#                         topology at 64-px resolution
#                       • dilation=1    – fine loop curvature / stroke width
#                       • dilation=2    – loop + immediately adjacent strokes;
#                         Thai consonants differ primarily in loop placement
#                         relative to the stem, so medium-range context matters
#                         more than very long-range (unlike Kuzushiji)
#    • Augmentation  : rotation ±8° KEPT (Thai handwriting varies in angle)
#                       random_flip REMOVED – Thai script is directional
#    • AFC           : OPTIONAL – 54 classes is manageable without it;
#                       included but with a lightweight capsule_dim=8
#    • Model name    : WhatNet-Kuzushiji → WhatNet-Thai
#    • Output files  : kuzushiji_*.json  → thai_*.json
#
#  Dataset
#  ───────
#  Burapha-TH  (Thai handwritten character dataset, Burapha University)
#    • ~60,000 64×64 greyscale images
#    • 54 classes: 44 Thai consonants + 10 Thai/Arabic digit classes
#    • Expected layout in CFG['data_dir']:
#        thai_train.npz  –  arr_0: uint8 (N, 64, 64),  arr_1: uint8 (N,)
#        thai_test.npz   –  arr_0: uint8 (M, 64, 64),  arr_1: uint8 (M,)
#      OR split files:
#        thai-train-imgs.npz / thai-train-lbls.npz
#        thai-test-imgs.npz  / thai-test-lbls.npz
#    • Kaggle path (if using the Burapha-TH Kaggle dataset):
#        "/kaggle/input/burapha-th/"
# ─────────────────────────────────────────────────────────────────────────────
import os, time, random, json
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# ─────────────────────────────────────────────────────────────────────────────
#  0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  1. CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    # 44 Thai consonants + 10 digit classes
    "num_classes":      68,
    # Burapha-TH is natively 64×64
    "image_size":       64,

    "batch_size":       64,
    "epochs":           50,
    "lr":               5e-4,
    "weight_decay":     1e-4,
    "label_smoothing":  0.05,
    "val_split":        0.1,

    # Path to Burapha-TH .npz files.
    # Kaggle notebook path example: "/kaggle/input/burapha-th/"
    "data_dir":    "/kaggle/input/burapha-th/",

    "results_dir": "./results",
}
os.makedirs(CFG["results_dir"], exist_ok=True)

NUM_CLASSES = CFG["num_classes"]
IMG         = CFG["image_size"]
BS          = CFG["batch_size"]
AUTOTUNE    = tf.data.AUTOTUNE

# ─────────────────────────────────────────────────────────────────────────────
#  2. DATASET PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
if not os.path.exists(CFG["data_dir"]):
    raise FileNotFoundError(
        f"[ERROR] Dataset directory not found: {CFG['data_dir']}\n"
        "Download Burapha-TH and place .npz files in the directory above."
    )

def _load_split_files(img_path: str, lbl_path: str):
    images = np.load(img_path)["arr_0"].astype(np.float32)
    labels = np.load(lbl_path)["arr_0"].astype(np.int32)
    return images, labels

def _load_combined_file(path: str):
    data   = np.load(path)
    images = data["arr_0"].astype(np.float32)
    labels = data["arr_1"].astype(np.int32)
    return images, labels

def _load_thai(split: str):
    """Auto-detect layout (split-file vs combined) and load accordingly."""
    d = CFG["data_dir"]
    # Layout A: split files
    img_path = os.path.join(d, f"thai-{split}-imgs.npz")
    lbl_path = os.path.join(d, f"thai-{split}-lbls.npz")
    if os.path.exists(img_path) and os.path.exists(lbl_path):
        print(f"[INFO] Loading {split} from split files: {img_path}, {lbl_path}")
        return _load_split_files(img_path, lbl_path)
    # Layout B: combined single file
    combined = os.path.join(d, f"thai_{split}.npz")
    if os.path.exists(combined):
        print(f"[INFO] Loading {split} from combined file: {combined}")
        return _load_combined_file(combined)
    raise FileNotFoundError(
        f"[ERROR] Could not find {split} data for Burapha-TH.\n"
        f"  Looked for (split-file): {img_path}, {lbl_path}\n"
        f"  Looked for (combined)  : {combined}\n"
        "Please verify the .npz files are in CFG['data_dir']."
    )

# ── Load raw arrays ───────────────────────────────────────────────────────────
x_train_all, y_train_all = _load_thai("train")
x_test,      y_test      = _load_thai("test")

print(f"[INFO] Unique classes in dataset : {len(np.unique(y_train_all))}")
print(f"[INFO] CFG num_classes           : {NUM_CLASSES}")
if len(np.unique(y_train_all)) != NUM_CLASSES:
    print(f"[WARN] Mismatch – update CFG['num_classes'] to "
          f"{len(np.unique(y_train_all))} to match your dataset.")

# ── Train / val split ─────────────────────────────────────────────────────────
n_total = len(x_train_all)
n_val   = max(1, int(n_total * CFG["val_split"]))
n_train = n_total - n_val

rng  = np.random.default_rng(SEED)
perm = rng.permutation(n_total)
x_train_all, y_train_all = x_train_all[perm], y_train_all[perm]

x_train, y_train = x_train_all[:n_train], y_train_all[:n_train]
x_val,   y_val   = x_train_all[n_train:], y_train_all[n_train:]

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {len(x_test):,}")

# ── Add channel dim: (N,64,64) → (N,64,64,1) ─────────────────────────────────
x_train = x_train[..., np.newaxis]
x_val   = x_val[...,   np.newaxis]
x_test  = x_test[...,  np.newaxis]

# ── Rotation augmentation ─────────────────────────────────────────────────────
_HAS_RANDOM_ROTATION = hasattr(tf.keras.layers, "RandomRotation")
if _HAS_RANDOM_ROTATION:
    print("[INFO] RandomRotation available – rotation augmentation enabled (±8°).")
    _rotator = tf.keras.layers.RandomRotation(
        factor=8/360,
        fill_mode="constant",
        fill_value=-1.0,
        seed=SEED,
    )
else:
    print("[WARN] RandomRotation not available – rotation augmentation disabled.")
    _rotator = None

# ── Preprocessing helpers ─────────────────────────────────────────────────────
def normalise(img, lbl):
    img = tf.cast(img, tf.float32) / 127.5 - 1.0
    return img, lbl

def augment(img, lbl):
    """
    Stochastic augmentation – training only.

    Thai consonants have:
      • Circular loops that sit at varying heights/positions
      • Diacritics (sara) that hover above or below the baseline
      • Significant pen-angle variation in handwriting

    Augmentations:
      • Brightness / contrast jitter – ink variation
      • Pad-then-random-crop         – translation ±4 px (64-px image)
      • Random rotation ±8°          – writing angle variation

    No flip: Thai script is strongly directional – the loop always curls
    inward in a consistent direction; mirroring would produce invalid glyphs.
    """
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    # ±4 px translation on 64-px images (pad=4 each side → crop back to 64)
    img = tf.pad(img, [[4, 4], [4, 4], [0, 0]], constant_values=-1.0)
    img = tf.image.random_crop(img, [IMG, IMG, 1])
    if _rotator is not None:
        img = _rotator(tf.expand_dims(img, 0))[0]
    return img, lbl

def to_onehot(img, lbl):
    return img, tf.one_hot(lbl, NUM_CLASSES)

# ── Build tf.data pipelines ───────────────────────────────────────────────────
train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .map(augment,   num_parallel_calls=AUTOTUNE)
    .map(to_onehot, num_parallel_calls=AUTOTUNE)
    .shuffle(8192, seed=SEED)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .map(to_onehot, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds_oh = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)

# ─────────────────────────────────────────────────────────────────────────────
#  3. DISPLAY UTILITIES
# ─────────────────────────────────────────────────────────────────────────────
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"

def print_model_summary(model: Model) -> None:
    W             = 62
    trainable     = model.count_params()
    non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
    total         = trainable + non_trainable
    title         = f"  {model.name}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╦{'═'*23}╦{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Layer':<16}║  {'Type':<21}║  {'Params':>15}  ║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╬{'═'*23}╬{'═'*18}╣", "cyan"))
    for lyr in [l for l in model.layers if l.count_params() > 0][:20]:
        n_p = lyr.count_params()
        print(f"║  {lyr.name[:14]:<16}║  {type(lyr).__name__[:21]:<21}║  {n_p:>15,}  ║")
    if len([l for l in model.layers if l.count_params() > 0]) > 20:
        print(f"║  {'… (truncated)':<16}║  {'':21}║  {'':>15}  ║")
    print(_c(f"╠{'═'*18}╩{'═'*23}╩{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {non_trainable:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))

class EpochProgressCallback(keras.callbacks.Callback):
    BAR_WIDTH = 20
    def __init__(self, total_epochs: int, model_name: str):
        super().__init__()
        self.total_epochs = total_epochs
        self.model_name   = model_name
        self._epoch_start = 0.0

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        logs    = logs or {}
        elapsed = time.time() - self._epoch_start
        ep_num  = epoch + 1
        pct     = ep_num / self.total_epochs
        filled  = int(self.BAR_WIDTH * pct)
        bar     = "█" * filled + "░" * (self.BAR_WIDTH - filled)
        loss    = logs.get("loss",         float("nan"))
        acc     = logs.get("accuracy",     float("nan")) * 100
        val_acc = logs.get("val_accuracy", float("nan")) * 100
        val_los = logs.get("val_loss",     float("nan"))
        try:
            lr_val = float(keras.backend.get_value(self.model.optimizer.learning_rate))
            lr_str = f"lr={lr_val:.2e}"
        except Exception:
            lr_str = ""
        print(
            f"  {_c(f'Epoch {ep_num:>3}/{self.total_epochs}', 'grey')}  "
            f"{_c(bar, 'cyan')} {_c(f'{pct*100:>5.1f}%', 'yellow')}  "
            f"{_c(f'loss={loss:.4f}', 'white')}  {_c(f'acc={acc:.2f}%', 'green')}  "
            f"{_c(f'val_loss={val_los:.4f}', 'white')}  "
            f"{_c(f'val_acc={val_acc:.2f}%', 'yellow' if val_acc < acc else 'green')}  "
            f"{_c(lr_str, 'blue')}  {_c(f'[{elapsed:.1f}s]', 'grey')}"
        )

def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║",
             "cyan", "bold"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n",
             "green", "bold"))

# ─────────────────────────────────────────────────────────────────────────────
#  4. BUILDING BLOCKS
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x):
    return tf.nn.gelu(x)

def residual_block(x, channels: int):
    residual = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, residual])
    x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_channels: int, out_channels: int):
    if in_channels != out_channels:
        skip = layers.Conv2D(out_channels, 1, use_bias=False)(x)
        skip = layers.BatchNormalization()(skip)
        x_in = layers.Activation(gelu)(skip)
    else:
        x_in = x
    r1  = residual_block(x_in, out_channels)
    r2  = residual_block(r1,   out_channels)
    r3  = residual_block(r2,   out_channels)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_channels, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_channels, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    return out

def channel_attention(x, channels: int, reduction: int = 8):
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(channels // reduction, activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def adaptive_filter_capsule(x, num_classes: int, capsule_dim: int = 8):
    """
    Lightweight AFC for Thai (54 classes, capsule_dim=8).

    Thai has only 54 classes so a slim capsule is sufficient; reducing
    capsule_dim from 16→8 cuts parameters while retaining per-class routing.
    """
    h        = layers.Dense(256, activation=gelu)(x)
    h        = layers.Dense(num_classes * capsule_dim)(h)
    h        = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps     = layers.Multiply()([x_sliced, h])
    caps     = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    caps     = layers.BatchNormalization()(caps)
    return caps

def stroke_topology_module(x, out_features: int):
    """
    Thai stroke topology encoder.

    Thai consonants are uniquely defined by:
      • A single circular or semi-circular loop at the glyph apex
        (distinguishes almost every consonant from every other)
      • A vertical main stem (สระ – sara vowel markers ride this stem)
      • Ascending/descending ligatures for vowel diacritics

    Kernel design:
      • 5×5 standard    – large enough to capture a full loop at 64-px
                           resolution; Thai loops span ≈10–20 px and a 3×3
                           kernel would see only a local arc fragment
      • dilation=1      – fine loop closure detection (adjacent pixels
                           completing the circle); critical for ก vs. ถ
      • dilation=2      – loop-to-stem relationship; doubles the 5×5
                           receptive field to ≈17 px, covering the loop and
                           the immediate stem region simultaneously

    No asymmetric axis-aligned kernels (no 1×N or N×1):
    Thai loop geometry is radially symmetric – thin horizontal or vertical
    strips would capture only part of the circular topology without the
    complementary axis, leading to ambiguous partial-arc features.
    """
    # 5×5 captures full loop at 64-px resolution
    loop    = layers.Conv2D(64, 5, padding="same", use_bias=False, activation=gelu)(x)
    # dilation=1 – fine local arc curvature
    fine    = layers.Conv2D(32, 5, padding="same", dilation_rate=1,
                            use_bias=False, activation=gelu)(x)
    # dilation=2 – loop + stem spatial context
    context = layers.Conv2D(32, 5, padding="same", dilation_rate=2,
                            use_bias=False, activation=gelu)(x)
    out  = layers.Concatenate()([loop, fine, context])
    out  = layers.BatchNormalization()(out)
    out  = layers.GlobalAveragePooling2D()(out)
    out  = layers.Dense(out_features, activation=gelu)(out)
    return out

def cross_scale_transformer_bridge(s1, s2, s3, dim: int = 256, num_heads: int = 4):
    t1       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s1))[:, tf.newaxis, :]
    t2       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s2))[:, tf.newaxis, :]
    t3       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s3))[:, tf.newaxis, :]
    seq      = layers.Concatenate(axis=1)([t1, t2, t3])
    attn_out = layers.MultiHeadAttention(num_heads=num_heads, key_dim=dim // num_heads)(seq, seq)
    attn_out = layers.LayerNormalization()(attn_out + seq)
    pooled   = layers.Lambda(lambda t: tf.reduce_mean(t, axis=1))(attn_out)
    pooled   = layers.Dense(dim, activation=gelu)(pooled)
    return pooled

# ─────────────────────────────────────────────────────────────────────────────
#  5. MODEL DEFINITION
# ─────────────────────────────────────────────────────────────────────────────
def build_whatnet_thai(num_classes: int = 54, image_size: int = 64) -> Model:
    """
    WhatNet-Thai: WhatNet adapted for Thai Burapha-TH recognition.

    Architecture overview
    ─────────────────────
    Stem (triple-path):
      • Standard 3×3 conv path       – general glyph body
      • 1×5 horizontal path          – Sara vowel diacritics (horizontal
                                        thin strokes above/below baseline)
      • 5×1 vertical path            – Consonant vertical main stems and
                                        descenders
      → Concatenated and refined with SE channel attention.
      Scaffold bypass is REMOVED: Thai circular loops require full encoder
      depth; a shallow bypass would produce imprecise loop representations.

    Encoder (3 stages, each halving spatial dims):
      enc1: 96→96    (32×32 at 64-px input)
      enc2: 96→192   (16×16)
      enc3: 192→384  ( 8× 8)
      No scaffold residuals (bypass removed).

    Decoder head:
      • Cross-scale transformer bridge (CSTB) – fine loop (d1) and
        loop-stem relationship (d2) interact across scales
      • Lightweight Adaptive Filter Capsule (AFC, capsule_dim=8)
      • Stroke topology module (STM) – 5×5 + dil(1,2)
      • Gated fusion → MLP + layer norm → logits (54 classes)
    """
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # ── Stem: triple-path (3×3, 1×5, 5×1) ────────────────────────────────
    t = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t = layers.BatchNormalization()(t)
    t = layers.Activation(gelu)(t)

    # 1×5 horizontal – diacritic horizontal strokes
    h = layers.Conv2D(32, (1, 5), padding="same", use_bias=False)(inp)
    h = layers.BatchNormalization()(h)
    h = layers.Activation(gelu)(h)

    # 5×1 vertical – consonant stems / descenders
    v = layers.Conv2D(32, (5, 1), padding="same", use_bias=False)(inp)
    v = layers.BatchNormalization()(v)
    v = layers.Activation(gelu)(v)

    stem = layers.Concatenate()([t, h, v])          # 96 channels
    stem = channel_attention(stem, 96)
    stem = layers.Conv2D(96, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem)
    stem = layers.Activation(gelu)(stem)

    # ── Encoder (no scaffold bypass) ──────────────────────────────────────
    enc1 = dense_res_block(stem, 96,  96)   # → 32×32
    enc2 = dense_res_block(enc1, 96,  192)  # → 16×16
    enc3 = dense_res_block(enc2, 192, 384)  # →  8× 8

    # ── Decoder head ──────────────────────────────────────────────────────
    cstb_out = cross_scale_transformer_bridge(enc1, enc2, enc3, dim=256)
    afc_in   = layers.GlobalAveragePooling2D()(enc3)
    afc_in   = layers.Add()([afc_in, cstb_out])
    afc_out  = adaptive_filter_capsule(afc_in, K, capsule_dim=8)

    stgm_out    = stroke_topology_module(enc3, 256)
    combined    = layers.Concatenate()([stgm_out, afc_out])
    gate        = layers.Dense(2, activation="softmax", name="gate")(combined)
    stgm_scaled = layers.Lambda(lambda t: t[0] * t[1][:, 0:1])([stgm_out, gate])
    fused       = layers.Concatenate()([stgm_scaled, afc_out])

    x   = layers.Dense(512)(fused)
    x   = layers.LayerNormalization()(x)
    x   = layers.Activation(gelu)(x)
    out = layers.Dense(K, name="logits")(x)

    return Model(inputs=inp, outputs=out, name="WhatNet-Thai")

MODELS_TF = {
    "WhatNet-Thai": lambda: build_whatnet_thai(NUM_CLASSES, IMG),
}

# ─────────────────────────────────────────────────────────────────────────────
#  6. LR SCHEDULE
# ─────────────────────────────────────────────────────────────────────────────
class CosineAnnealing(keras.optimizers.schedules.LearningRateSchedule):
    """Cosine-annealing without restarts: lr(t) = max(base·½·(1+cos(π·t/T)), 1e-6)."""
    def __init__(self, base: float, steps: int):
        self.base  = base
        self.steps = tf.cast(steps, tf.float32)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        cosine = 0.5 * (1.0 + tf.cos(np.pi * step / self.steps))
        return tf.maximum(self.base * cosine, 1e-6)

    def get_config(self):
        return {"base": self.base, "steps": int(self.steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  7. TRAINING & EVALUATION HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def compile_model(model: Model, steps_total: int) -> Model:
    lr_sch = CosineAnnealing(CFG["lr"], steps_total)
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=lr_sch,
            weight_decay=CFG["weight_decay"],
        ),
        loss=keras.losses.CategoricalCrossentropy(
            from_logits=True,
            label_smoothing=CFG["label_smoothing"],
        ),
        metrics=["accuracy"],
        jit_compile=True,
    )
    return model

def compute_macro_f1(model: Model, dataset) -> float:
    """Macro-averaged F1 over all NUM_CLASSES classes (returns %)."""
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for images, labels in dataset:
        preds = tf.argmax(model(images, training=False), axis=1).numpy()
        lbls  = labels.numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

# ─────────────────────────────────────────────────────────────────────────────
#  8. TRAIN + EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
trained_models  = {}
all_histories   = {}
steps_per_epoch = -(-n_train // BS)
total_steps     = steps_per_epoch * CFG["epochs"]

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting benchmark: {len(MODELS_TF)} model(s) × {CFG['epochs']} epochs"
         f"  [Burapha-TH Thai]", "bold"))
print(_c(f"{'═'*70}\n", "cyan"))

for name, model_fn in MODELS_TF.items():
    model = model_fn()
    model = compile_model(model, total_steps)
    print_model_summary(model)

    ckpt_path = os.path.join(CFG["results_dir"], f"{name}_best.keras")
    callbacks = [
        ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=15, restore_best_weights=True, verbose=1),
        # EpochProgressCallback(CFG["epochs"], name),
    ]

    print(f"\n{_c('  ▶ Training:', 'bold', 'cyan')} {_c(name, 'yellow')}")
    t0      = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CFG["epochs"],
        callbacks=callbacks,
        verbose=1,
    )
    elapsed  = time.time() - t0
    best_val = max(history.history["val_accuracy"]) * 100.0
    print(
        f"\n  {_c('✔ Done:', 'green', 'bold')} "
        f"best val acc = {_c(f'{best_val:.2f}%', 'green')}  "
        f"wall time = {_c(f'{elapsed:.0f}s', 'grey')}"
    )
    trained_models[name] = model
    all_histories[name]  = history.history

# ─────────────────────────────────────────────────────────────────────────────
#  9. FINAL TEST-SET EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
results = {}
for name, model in trained_models.items():
    test_loss, test_acc_raw = model.evaluate(test_ds_oh, verbose=0)
    test_acc  = test_acc_raw * 100.0
    macro_f1  = compute_macro_f1(model, test_ds)
    results[name] = {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    model.count_params(),
        "test_loss": round(float(test_loss), 4),
    }
print_comparison_table(results)

# ─────────────────────────────────────────────────────────────────────────────
#  10. PERSIST RESULTS
# ─────────────────────────────────────────────────────────────────────────────
results_path = os.path.join(CFG["results_dir"], "thai_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results   → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "thai_histories.json")
with open(histories_path, "w") as f:
    json.dump(
        {n: {k: [float(v) for v in vals] for k, vals in h.items()}
         for n, h in all_histories.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] Burapha-TH Thai benchmark complete.\n", "green", "bold"))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  WhatNet  –  Thai Script Recognition  (folder-based dataset)
#
#  Dataset layout expected:
#    thai-dataset/
#      train/
#        00-161-A1-KO KAI/        ← folder name = class label
#          img001.png
#          img002.png
#          ...
#        01-162-A2-KHO KHAI/
#          ...
#      test/
#        00-161-A1-KO KAI/
#          ...
#
#  Labels are derived from the folder sort order (0-indexed),
#  so train and test folders must have identical class ordering.
# ─────────────────────────────────────────────────────────────────────────────
import os, time, random, json
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# ─────────────────────────────────────────────────────────────────────────────
#  0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  1. CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    # Will be auto-detected from the number of class folders found on disk.
    # Override here only if you want to force a specific value.
    "num_classes":      68,       # None = auto-detect
    "image_size":       64,
    "batch_size":       64,
    "epochs":           50,
    "lr":               5e-4,
    "weight_decay":     1e-4,
    "label_smoothing":  0.05,
    "val_split":        0.1,

    # Root folder that contains "train/" and "test/" sub-directories.
    # Kaggle path example: "/kaggle/input/thai-dataset"
    "data_dir":    "/kaggle/input/datasets/rautranjit/thai-dataset",

    "results_dir": "./results",
}
os.makedirs(CFG["results_dir"], exist_ok=True)

IMG     = CFG["image_size"]
BS      = CFG["batch_size"]
AUTOTUNE = tf.data.AUTOTUNE

# ─────────────────────────────────────────────────────────────────────────────
#  2. DATASET PIPELINE  (folder-image loader)
# ─────────────────────────────────────────────────────────────────────────────
train_dir = os.path.join(CFG["data_dir"], "train")
test_dir  = os.path.join(CFG["data_dir"], "test")

for d in (train_dir, test_dir):
    if not os.path.isdir(d):
        raise FileNotFoundError(
            f"[ERROR] Expected directory not found: {d}\n"
            f"Make sure CFG['data_dir'] points to the folder that contains "
            f"'train/' and 'test/' sub-directories."
        )

# ── Auto-detect num_classes from folder count ─────────────────────────────────
_train_classes = sorted([
    c for c in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, c))
])
_test_classes  = sorted([
    c for c in os.listdir(test_dir)
    if os.path.isdir(os.path.join(test_dir, c))
])

if CFG["num_classes"] is None:
    NUM_CLASSES = len(_train_classes)
    print(f"[INFO] Auto-detected {NUM_CLASSES} classes from train folder.")
else:
    NUM_CLASSES = CFG["num_classes"]
    print(f"[INFO] Using CFG num_classes = {NUM_CLASSES}.")

if len(_train_classes) != len(_test_classes):
    print(f"[WARN] train has {len(_train_classes)} class folders but "
          f"test has {len(_test_classes)} – labels may be inconsistent.")

if len(_train_classes) != NUM_CLASSES:
    print(f"[WARN] Folder count ({len(_train_classes)}) != NUM_CLASSES "
          f"({NUM_CLASSES}).  Updating NUM_CLASSES.")
    NUM_CLASSES = len(_train_classes)

print(f"[INFO] Classes (first 5): {_train_classes[:5]} …")

# ── Helper: load a split from an image folder ────────────────────────────────
def load_folder_split(root: str, class_names: list) -> tuple:
    """
    Walk root/<class_name>/* and return (images_uint8, labels_int32).

    Images are resized to (IMG, IMG) and converted to greyscale.
    Labels are the sorted folder index (0-based).
    """
    class_to_idx = {c: i for i, c in enumerate(class_names)}
    images, labels = [], []

    for cls in class_names:
        cls_dir = os.path.join(root, cls)
        if not os.path.isdir(cls_dir):
            print(f"[WARN] Missing class folder: {cls_dir}")
            continue
        idx = class_to_idx[cls]
        for fname in sorted(os.listdir(cls_dir)):
            fpath = os.path.join(cls_dir, fname)
            if not os.path.isfile(fpath):
                continue
            ext = os.path.splitext(fname)[1].lower()
            if ext not in {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}:
                continue
            try:
                raw    = tf.io.read_file(fpath)
                # decode_image handles PNG / JPEG / BMP / GIF automatically
                img    = tf.image.decode_image(raw, channels=1, expand_animations=False)
                img    = tf.image.resize(img, [IMG, IMG])
                img    = tf.cast(img, tf.uint8)
                images.append(img.numpy())
                labels.append(idx)
            except Exception as e:
                print(f"[WARN] Could not read {fpath}: {e}")

    images_np = np.array(images, dtype=np.uint8)   # (N, IMG, IMG, 1)
    labels_np = np.array(labels, dtype=np.int32)
    return images_np, labels_np

print("[INFO] Loading train images …  (this may take a moment)")
x_train_all, y_train_all = load_folder_split(train_dir, _train_classes)
print(f"[INFO] Train loaded: {len(x_train_all):,} images")

print("[INFO] Loading test images …")
x_test, y_test = load_folder_split(test_dir, _train_classes)
print(f"[INFO] Test  loaded: {len(x_test):,} images")

# ── Squeeze channel dim for val split (re-added later) ───────────────────────
# images are currently (N, IMG, IMG, 1)  – squeeze to (N, IMG, IMG) for split
x_train_all = x_train_all[..., 0]   # → (N, IMG, IMG)
x_test      = x_test[..., 0]         # → (N, IMG, IMG)

# ── Train / val split ─────────────────────────────────────────────────────────
n_total = len(x_train_all)
n_val   = max(1, int(n_total * CFG["val_split"]))
n_train = n_total - n_val

rng  = np.random.default_rng(SEED)
perm = rng.permutation(n_total)
x_train_all, y_train_all = x_train_all[perm], y_train_all[perm]

x_train, y_train = x_train_all[:n_train], y_train_all[:n_train]
x_val,   y_val   = x_train_all[n_train:], y_train_all[n_train:]

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {len(x_test):,}")

# ── Add channel dim: (N,IMG,IMG) → (N,IMG,IMG,1) ─────────────────────────────
x_train = x_train[..., np.newaxis]
x_val   = x_val[...,   np.newaxis]
x_test  = x_test[...,  np.newaxis]

# ── Rotation augmentation ─────────────────────────────────────────────────────
_HAS_RANDOM_ROTATION = hasattr(tf.keras.layers, "RandomRotation")
if _HAS_RANDOM_ROTATION:
    print("[INFO] RandomRotation available – rotation augmentation enabled (±8°).")
    _rotator = tf.keras.layers.RandomRotation(
        factor=8/360,
        fill_mode="constant",
        fill_value=-1.0,
        seed=SEED,
    )
else:
    print("[WARN] RandomRotation not available – rotation augmentation disabled.")
    _rotator = None

# ── Preprocessing helpers ─────────────────────────────────────────────────────
def normalise(img, lbl):
    img = tf.cast(img, tf.float32) / 127.5 - 1.0
    return img, lbl

def augment(img, lbl):
    """
    Stochastic augmentation – training only.

    Thai consonants have:
      • Circular loops that sit at varying heights/positions
      • Diacritics (sara) that hover above or below the baseline
      • Significant pen-angle variation in handwriting

    Augmentations:
      • Brightness / contrast jitter – ink variation
      • Pad-then-random-crop         – translation ±4 px (64-px image)
      • Random rotation ±8°          – writing angle variation

    No flip: Thai script is strongly directional.
    """
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    # ±4 px translation on 64-px images (pad=4 each side → crop back to 64)
    img = tf.pad(img, [[4, 4], [4, 4], [0, 0]], constant_values=-1.0)
    img = tf.image.random_crop(img, [IMG, IMG, 1])
    if _rotator is not None:
        img = _rotator(tf.expand_dims(img, 0))[0]
    return img, lbl

def to_onehot(img, lbl):
    return img, tf.one_hot(lbl, NUM_CLASSES)

# ── Build tf.data pipelines ───────────────────────────────────────────────────
train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .map(augment,   num_parallel_calls=AUTOTUNE)
    .map(to_onehot, num_parallel_calls=AUTOTUNE)
    .shuffle(8192, seed=SEED)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .map(to_onehot, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds_oh = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)

# ─────────────────────────────────────────────────────────────────────────────
#  3. DISPLAY UTILITIES
# ─────────────────────────────────────────────────────────────────────────────
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"

def print_model_summary(model: Model) -> None:
    W             = 62
    trainable     = model.count_params()
    non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
    total         = trainable + non_trainable
    title         = f"  {model.name}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╦{'═'*23}╦{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Layer':<16}║  {'Type':<21}║  {'Params':>15}  ║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╬{'═'*23}╬{'═'*18}╣", "cyan"))
    for lyr in [l for l in model.layers if l.count_params() > 0][:20]:
        n_p = lyr.count_params()
        print(f"║  {lyr.name[:14]:<16}║  {type(lyr).__name__[:21]:<21}║  {n_p:>15,}  ║")
    if len([l for l in model.layers if l.count_params() > 0]) > 20:
        print(f"║  {'… (truncated)':<16}║  {'':21}║  {'':>15}  ║")
    print(_c(f"╠{'═'*18}╩{'═'*23}╩{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {non_trainable:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))

class EpochProgressCallback(keras.callbacks.Callback):
    BAR_WIDTH = 20
    def __init__(self, total_epochs: int, model_name: str):
        super().__init__()
        self.total_epochs = total_epochs
        self.model_name   = model_name
        self._epoch_start = 0.0

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        logs    = logs or {}
        elapsed = time.time() - self._epoch_start
        ep_num  = epoch + 1
        pct     = ep_num / self.total_epochs
        filled  = int(self.BAR_WIDTH * pct)
        bar     = "█" * filled + "░" * (self.BAR_WIDTH - filled)
        loss    = logs.get("loss",         float("nan"))
        acc     = logs.get("accuracy",     float("nan")) * 100
        val_acc = logs.get("val_accuracy", float("nan")) * 100
        val_los = logs.get("val_loss",     float("nan"))
        try:
            lr_val = float(keras.backend.get_value(self.model.optimizer.learning_rate))
            lr_str = f"lr={lr_val:.2e}"
        except Exception:
            lr_str = ""
        print(
            f"  {_c(f'Epoch {ep_num:>3}/{self.total_epochs}', 'grey')}  "
            f"{_c(bar, 'cyan')} {_c(f'{pct*100:>5.1f}%', 'yellow')}  "
            f"{_c(f'loss={loss:.4f}', 'white')}  {_c(f'acc={acc:.2f}%', 'green')}  "
            f"{_c(f'val_loss={val_los:.4f}', 'white')}  "
            f"{_c(f'val_acc={val_acc:.2f}%', 'yellow' if val_acc < acc else 'green')}  "
            f"{_c(lr_str, 'blue')}  {_c(f'[{elapsed:.1f}s]', 'grey')}"
        )

def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║",
             "cyan", "bold"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n",
             "green", "bold"))

# ─────────────────────────────────────────────────────────────────────────────
#  4. BUILDING BLOCKS
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x):
    return tf.nn.gelu(x)

def residual_block(x, channels: int):
    residual = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, residual])
    x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_channels: int, out_channels: int):
    if in_channels != out_channels:
        skip = layers.Conv2D(out_channels, 1, use_bias=False)(x)
        skip = layers.BatchNormalization()(skip)
        x_in = layers.Activation(gelu)(skip)
    else:
        x_in = x
    r1  = residual_block(x_in, out_channels)
    r2  = residual_block(r1,   out_channels)
    r3  = residual_block(r2,   out_channels)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_channels, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_channels, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    return out

def channel_attention(x, channels: int, reduction: int = 8):
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(channels // reduction, activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def adaptive_filter_capsule(x, num_classes: int, capsule_dim: int = 8):
    h        = layers.Dense(256, activation=gelu)(x)
    h        = layers.Dense(num_classes * capsule_dim)(h)
    h        = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps     = layers.Multiply()([x_sliced, h])
    caps     = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    caps     = layers.BatchNormalization()(caps)
    return caps

def stroke_topology_module(x, out_features: int):
    loop    = layers.Conv2D(64, 5, padding="same", use_bias=False, activation=gelu)(x)
    fine    = layers.Conv2D(32, 5, padding="same", dilation_rate=1,
                            use_bias=False, activation=gelu)(x)
    context = layers.Conv2D(32, 5, padding="same", dilation_rate=2,
                            use_bias=False, activation=gelu)(x)
    out  = layers.Concatenate()([loop, fine, context])
    out  = layers.BatchNormalization()(out)
    out  = layers.GlobalAveragePooling2D()(out)
    out  = layers.Dense(out_features, activation=gelu)(out)
    return out

def cross_scale_transformer_bridge(s1, s2, s3, dim: int = 256, num_heads: int = 4):
    t1       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s1))[:, tf.newaxis, :]
    t2       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s2))[:, tf.newaxis, :]
    t3       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s3))[:, tf.newaxis, :]
    seq      = layers.Concatenate(axis=1)([t1, t2, t3])
    attn_out = layers.MultiHeadAttention(num_heads=num_heads, key_dim=dim // num_heads)(seq, seq)
    attn_out = layers.LayerNormalization()(attn_out + seq)
    pooled   = layers.Lambda(lambda t: tf.reduce_mean(t, axis=1))(attn_out)
    pooled   = layers.Dense(dim, activation=gelu)(pooled)
    return pooled

# ─────────────────────────────────────────────────────────────────────────────
#  5. MODEL DEFINITION
# ─────────────────────────────────────────────────────────────────────────────
def build_whatnet_thai(num_classes: int, image_size: int = 64) -> Model:
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # ── Stem: triple-path (3×3, 1×5, 5×1) ────────────────────────────────
    t = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t = layers.BatchNormalization()(t)
    t = layers.Activation(gelu)(t)

    h = layers.Conv2D(32, (1, 5), padding="same", use_bias=False)(inp)
    h = layers.BatchNormalization()(h)
    h = layers.Activation(gelu)(h)

    v = layers.Conv2D(32, (5, 1), padding="same", use_bias=False)(inp)
    v = layers.BatchNormalization()(v)
    v = layers.Activation(gelu)(v)

    stem = layers.Concatenate()([t, h, v])          # 96 channels
    stem = channel_attention(stem, 96)
    stem = layers.Conv2D(96, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem)
    stem = layers.Activation(gelu)(stem)

    # ── Encoder (no scaffold bypass) ──────────────────────────────────────
    enc1 = dense_res_block(stem, 96,  96)   # → 32×32
    enc2 = dense_res_block(enc1, 96,  192)  # → 16×16
    enc3 = dense_res_block(enc2, 192, 384)  # →  8× 8

    # ── Decoder head ──────────────────────────────────────────────────────
    cstb_out = cross_scale_transformer_bridge(enc1, enc2, enc3, dim=256)
    afc_in   = layers.GlobalAveragePooling2D()(enc3)        # (B, 384)
    afc_in   = layers.Dense(256, use_bias=False)(afc_in)    # (B, 256) – match cstb_out
    afc_in   = layers.Add()([afc_in, cstb_out])
    afc_out  = adaptive_filter_capsule(afc_in, K, capsule_dim=8)

    stgm_out    = stroke_topology_module(enc3, 256)
    combined    = layers.Concatenate()([stgm_out, afc_out])
    gate        = layers.Dense(2, activation="softmax", name="gate")(combined)
    stgm_scaled = layers.Lambda(lambda t: t[0] * t[1][:, 0:1])([stgm_out, gate])
    fused       = layers.Concatenate()([stgm_scaled, afc_out])

    x   = layers.Dense(512)(fused)
    x   = layers.LayerNormalization()(x)
    x   = layers.Activation(gelu)(x)
    out = layers.Dense(K, name="logits")(x)

    return Model(inputs=inp, outputs=out, name="WhatNet-Thai")

MODELS_TF = {
    "WhatNet-Thai": lambda: build_whatnet_thai(NUM_CLASSES, IMG),
}

# ─────────────────────────────────────────────────────────────────────────────
#  6. LR SCHEDULE
# ─────────────────────────────────────────────────────────────────────────────
class CosineAnnealing(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base: float, steps: int):
        self.base  = base
        self.steps = tf.cast(steps, tf.float32)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        cosine = 0.5 * (1.0 + tf.cos(np.pi * step / self.steps))
        return tf.maximum(self.base * cosine, 1e-6)

    def get_config(self):
        return {"base": self.base, "steps": int(self.steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  7. TRAINING & EVALUATION HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def compile_model(model: Model, steps_total: int) -> Model:
    lr_sch = CosineAnnealing(CFG["lr"], steps_total)
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=lr_sch,
            weight_decay=CFG["weight_decay"],
        ),
        loss=keras.losses.CategoricalCrossentropy(
            from_logits=True,
            label_smoothing=CFG["label_smoothing"],
        ),
        metrics=["accuracy"],
        jit_compile=True,
    )
    return model

def compute_macro_f1(model: Model, dataset) -> float:
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for images, labels in dataset:
        preds = tf.argmax(model(images, training=False), axis=1).numpy()
        lbls  = labels.numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

# ─────────────────────────────────────────────────────────────────────────────
#  8. TRAIN + EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
trained_models  = {}
all_histories   = {}
steps_per_epoch = -(-n_train // BS)
total_steps     = steps_per_epoch * CFG["epochs"]

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting benchmark: {len(MODELS_TF)} model(s) × {CFG['epochs']} epochs"
         f"  [Thai folder dataset, {NUM_CLASSES} classes]", "bold"))
print(_c(f"{'═'*70}\n", "cyan"))

for name, model_fn in MODELS_TF.items():
    model = model_fn()
    model = compile_model(model, total_steps)
    print_model_summary(model)

    ckpt_path = os.path.join(CFG["results_dir"], f"{name}_best.keras")
    callbacks = [
        ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True, verbose=1),
        EpochProgressCallback(CFG["epochs"], name),
    ]

    print(f"\n{_c('  ▶ Training:', 'bold', 'cyan')} {_c(name, 'yellow')}")
    t0      = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CFG["epochs"],
        callbacks=callbacks,
        verbose=0,
    )
    elapsed  = time.time() - t0
    best_val = max(history.history["val_accuracy"]) * 100.0
    print(
        f"\n  {_c('✔ Done:', 'green', 'bold')} "
        f"best val acc = {_c(f'{best_val:.2f}%', 'green')}  "
        f"wall time = {_c(f'{elapsed:.0f}s', 'grey')}"
    )
    trained_models[name] = model
    all_histories[name]  = history.history

# ─────────────────────────────────────────────────────────────────────────────
#  9. FINAL TEST-SET EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
results = {}
for name, model in trained_models.items():
    test_loss, test_acc_raw = model.evaluate(test_ds_oh, verbose=0)
    test_acc  = test_acc_raw * 100.0
    macro_f1  = compute_macro_f1(model, test_ds)
    results[name] = {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    model.count_params(),
        "test_loss": round(float(test_loss), 4),
    }
print_comparison_table(results)

# ─────────────────────────────────────────────────────────────────────────────
#  10. PERSIST RESULTS
# ─────────────────────────────────────────────────────────────────────────────
results_path = os.path.join(CFG["results_dir"], "thai_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results   → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "thai_histories.json")
with open(histories_path, "w") as f:
    json.dump(
        {n: {k: [float(v) for v in vals] for k, vals in h.items()}
         for n, h in all_histories.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] Thai folder-dataset benchmark complete.\n", "green", "bold"))

2026-04-21 15:00:36.756917: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776783636.914955      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776783636.961109      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776783637.341379      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776783637.341415      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776783637.341417      55 computation_placer.cc:177] computation placer alr

[INFO] Using CFG num_classes = 68.
[INFO] Classes (first 5): ['00-161-A1-KO KAI', '01-162-A2-KHO KHAI', '02-163-A3-KHO KHUAT', '03-164-A4-KHO KHWAI', '04-165-A5-KHO KHON'] …
[INFO] Loading train images …  (this may take a moment)


I0000 00:00:1776783661.207094      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776783661.213052      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


[INFO] Train loaded: 63,327 images
[INFO] Loading test images …
[INFO] Test  loaded: 13,600 images
[INFO] Train: 56,995 | Val: 6,332 | Test: 13,600
[INFO] RandomRotation available – rotation augmentation enabled (±8°).

══════════════════════════════════════════════════════════════════════
  Starting benchmark: 1 model(s) × 50 epochs  [Thai folder dataset, 68 classes]
══════════════════════════════════════════════════════════════════════


╔══════════════════════════════════════════════════════════════╗
║  WhatNet-Thai  –  Parameter Summary                          ║
╠══════════════════╦═══════════════════════╦══════════════════╣
║  Layer           ║  Type                 ║           Params  ║
╠══════════════════╬═══════════════════════╬══════════════════╣
║  conv2d          ║  Conv2D               ║              288  ║
║  conv2d_1        ║  Conv2D               ║              160  ║
║  conv2d_2        ║  Conv2D               ║              160  ║
║  batch_normaliz  ║  BatchNormalizati

I0000 00:00:1776784823.299462     126 service.cc:152] XLA service 0x7aecd80028d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776784823.299502     126 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1776784823.299506     126 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1776784827.873646     126 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1776784863.184930     126 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  Epoch   1/50  ░░░░░░░░░░░░░░░░░░░░   2.0%  loss=4.2714  acc=1.45%  val_loss=4.2488  val_acc=1.58%  lr=5.00e-04  [556.5s]
  Epoch   2/50  ░░░░░░░░░░░░░░░░░░░░   4.0%  loss=4.2499  acc=1.47%  val_loss=4.2487  val_acc=1.48%  lr=4.98e-04  [440.6s]
  Epoch   3/50  █░░░░░░░░░░░░░░░░░░░   6.0%  loss=4.2412  acc=1.47%  val_loss=4.2405  val_acc=1.72%  lr=4.96e-04  [441.1s]
  Epoch   4/50  █░░░░░░░░░░░░░░░░░░░   8.0%  loss=4.2376  acc=1.50%  val_loss=4.2308  val_acc=1.47%  lr=4.92e-04  [440.1s]
  Epoch   5/50  ██░░░░░░░░░░░░░░░░░░  10.0%  loss=2.9306  acc=30.21%  val_loss=2.2331  val_acc=40.93%  lr=4.88e-04  [444.1s]
  Epoch   6/50  ██░░░░░░░░░░░░░░░░░░  12.0%  loss=0.9235  acc=83.99%  val_loss=1.1193  val_acc=77.76%  lr=4.82e-04  [445.7s]
  Epoch   7/50  ██░░░░░░░░░░░░░░░░░░  14.0%  loss=0.7420  acc=90.02%  val_loss=0.7751  val_acc=88.79%  lr=4.76e-04  [446.1s]
  Epoch   8/50  ███░░░░░░░░░░░░░░░░░  16.0%  loss=0.6720  acc=92.15%  val_loss=4.9859  val_acc=1.52%  lr=4.69e-04  [444.4s]
  Epoch  

# Flaw Method

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  WhatNet  –  Thai Script Recognition  (PyTorch, folder-based dataset)
#
#  Dataset layout expected:
#    thai-dataset/
#      train/
#        00-161-A1-KO KAI/        ← folder name = class label
#          img001.png
#          ...
#        01-162-A2-KHO KHAI/
#          ...
#      test/
#        00-161-A1-KO KAI/
#          ...
#
#  Labels are derived from folder sort order (0-indexed).
# ─────────────────────────────────────────────────────────────────────────────
import os, time, random, json, math
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

# ─────────────────────────────────────────────────────────────────────────────
#  0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {DEVICE}")

# ─────────────────────────────────────────────────────────────────────────────
#  1. CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    "num_classes":     68,       # None = auto-detect
    "image_size":      64,
    "batch_size":      64,
    "epochs":          50,
    "lr":              5e-4,
    "weight_decay":    1e-4,
    "label_smoothing": 0.05,
    "val_split":       0.1,

    "data_dir":    "/kaggle/input/datasets/rautranjit/thai-dataset",
    "results_dir": "./results",
}
os.makedirs(CFG["results_dir"], exist_ok=True)

IMG = CFG["image_size"]
BS  = CFG["batch_size"]

# ─────────────────────────────────────────────────────────────────────────────
#  2. DATASET
# ─────────────────────────────────────────────────────────────────────────────
train_dir = os.path.join(CFG["data_dir"], "train")
test_dir  = os.path.join(CFG["data_dir"], "test")

for d in (train_dir, test_dir):
    if not os.path.isdir(d):
        raise FileNotFoundError(
            f"[ERROR] Expected directory not found: {d}\n"
            f"Make sure CFG['data_dir'] points to the folder that contains "
            f"'train/' and 'test/' sub-directories."
        )

_train_classes = sorted([
    c for c in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, c))
])
_test_classes = sorted([
    c for c in os.listdir(test_dir)
    if os.path.isdir(os.path.join(test_dir, c))
])

if CFG["num_classes"] is None:
    NUM_CLASSES = len(_train_classes)
    print(f"[INFO] Auto-detected {NUM_CLASSES} classes from train folder.")
else:
    NUM_CLASSES = CFG["num_classes"]
    print(f"[INFO] Using CFG num_classes = {NUM_CLASSES}.")

if len(_train_classes) != len(_test_classes):
    print(f"[WARN] train has {len(_train_classes)} class folders but "
          f"test has {len(_test_classes)} – labels may be inconsistent.")

if len(_train_classes) != NUM_CLASSES:
    print(f"[WARN] Folder count ({len(_train_classes)}) != NUM_CLASSES "
          f"({NUM_CLASSES}). Updating NUM_CLASSES.")
    NUM_CLASSES = len(_train_classes)

print(f"[INFO] Classes (first 5): {_train_classes[:5]} …")

IMG_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}


class ThaiScriptDataset(Dataset):
    """
    Loads greyscale images from a folder hierarchy:
        root/<class_name>/<image_file>

    class_names must be the same sorted list for both train and test splits
    so that class indices are consistent.
    """

    def __init__(self, root: str, class_names: list, transform=None):
        self.transform    = transform
        self.class_to_idx = {c: i for i, c in enumerate(class_names)}
        self.samples      = []   # list of (filepath, label_int)

        for cls in class_names:
            cls_dir = os.path.join(root, cls)
            if not os.path.isdir(cls_dir):
                print(f"[WARN] Missing class folder: {cls_dir}")
                continue
            idx = self.class_to_idx[cls]
            for fname in sorted(os.listdir(cls_dir)):
                fpath = os.path.join(cls_dir, fname)
                if not os.path.isfile(fpath):
                    continue
                if os.path.splitext(fname)[1].lower() not in IMG_EXTENSIONS:
                    continue
                self.samples.append((fpath, idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        fpath, label = self.samples[index]
        try:
            img = Image.open(fpath).convert("L")   # greyscale
        except Exception as e:
            print(f"[WARN] Could not read {fpath}: {e}")
            img = Image.new("L", (IMG, IMG))
        if self.transform:
            img = self.transform(img)
        return img, label


# ── Transforms ────────────────────────────────────────────────────────────────
#  Normalise to [-1, 1]  (matches TF code: img / 127.5 - 1)
_MEAN = (0.5,)
_STD  = (0.5,)

train_transform = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.RandomAffine(
        degrees=8,                          # ±8° rotation
        translate=(4/64, 4/64),             # ±4 px translation on 64-px image
        fill=0,
    ),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),                  # → [0,1], shape (1, H, W)
    transforms.Normalize(_MEAN, _STD),      # → [-1, 1]
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),
    transforms.Normalize(_MEAN, _STD),
])

# ── Build datasets & loaders ──────────────────────────────────────────────────
print("[INFO] Indexing train images …")
full_train_ds = ThaiScriptDataset(train_dir, _train_classes, transform=train_transform)
test_ds_raw   = ThaiScriptDataset(test_dir,  _train_classes, transform=eval_transform)

n_total = len(full_train_ds)
n_val   = max(1, int(n_total * CFG["val_split"]))
n_train = n_total - n_val

gen = torch.Generator().manual_seed(SEED)
train_ds, val_ds = random_split(full_train_ds, [n_train, n_val], generator=gen)

# Val split was created from train_transform; swap to eval_transform for val
# The cleanest way: wrap with a view that overrides the transform.
class TransformOverride(Dataset):
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        fpath, label = self.subset.dataset.samples[self.subset.indices[idx]]
        try:
            img = Image.open(fpath).convert("L")
        except Exception:
            img = Image.new("L", (IMG, IMG))
        return self.transform(img), label

val_ds = TransformOverride(val_ds, eval_transform)

train_loader = DataLoader(train_ds,    batch_size=BS, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,      batch_size=BS, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds_raw, batch_size=BS, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {len(test_ds_raw):,}")

# ─────────────────────────────────────────────────────────────────────────────
#  3. DISPLAY UTILITIES
# ─────────────────────────────────────────────────────────────────────────────
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"

def count_params(model: nn.Module):
    trainable     = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    return trainable, non_trainable

def print_model_summary(model: nn.Module, name: str) -> None:
    W = 62
    trainable, non_trainable = count_params(model)
    total = trainable + non_trainable
    title = f"  {name}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╦{'═'*23}╦{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Layer':<16}║  {'Type':<21}║  {'Params':>15}  ║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╬{'═'*23}╬{'═'*18}╣", "cyan"))
    shown = 0
    for lname, module in model.named_modules():
        n_p = sum(p.numel() for p in module.parameters(recurse=False))
        if n_p > 0:
            shown += 1
            if shown <= 20:
                short = lname[-14:] if len(lname) > 14 else lname
                print(f"║  {short:<16}║  {type(module).__name__[:21]:<21}║  {n_p:>15,}  ║")
    if shown > 20:
        print(f"║  {'… (truncated)':<16}║  {'':21}║  {'':>15}  ║")
    print(_c(f"╠{'═'*18}╩{'═'*23}╩{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {non_trainable:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))

def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║",
             "cyan", "bold"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n",
             "green", "bold"))

# ─────────────────────────────────────────────────────────────────────────────
#  4. BUILDING BLOCKS
# ─────────────────────────────────────────────────────────────────────────────

class ResidualBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        x = F.gelu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x = F.gelu(x + residual)
        return x


class DenseResBlock(nn.Module):
    """
    3 stacked ResidualBlocks with dense concatenation, then
    a channel-collapse conv + strided depthwise conv for 2× downsampling.
    """
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        # Optional projection when channel dims differ
        if in_channels != out_channels:
            self.proj = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.proj = None

        self.r1 = ResidualBlock(out_channels)
        self.r2 = ResidualBlock(out_channels)
        self.r3 = ResidualBlock(out_channels)

        # Dense concat: 3 × out_channels → out_channels
        self.collapse = nn.Sequential(
            nn.Conv2d(3 * out_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        # Strided DW + PW for 2× spatial downsampling
        self.dw = nn.Conv2d(out_channels, out_channels, 3, stride=2,
                            padding=1, groups=out_channels, bias=False)
        self.pw = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        if self.proj is not None:
            x = F.gelu(self.proj(x))

        r1 = self.r1(x)
        r2 = self.r2(r1)
        r3 = self.r3(r2)

        cat = torch.cat([r1, r2, r3], dim=1)
        out = F.gelu(self.collapse(cat))
        out = F.gelu(self.pw(self.dw(out)))
        return out


class ChannelAttention(nn.Module):
    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)

    def forward(self, x):
        # x: (B, C, H, W)
        gap  = x.mean(dim=[2, 3])                    # (B, C)
        attn = F.relu(self.fc1(gap))
        attn = torch.sigmoid(self.fc2(attn))         # (B, C)
        return x * attn.unsqueeze(-1).unsqueeze(-1)


class AdaptiveFilterCapsule(nn.Module):
    """
    Generates per-class capsule activations from a global feature vector.
    """
    def __init__(self, in_features: int, num_classes: int, capsule_dim: int = 8):
        super().__init__()
        self.num_classes = num_classes
        self.capsule_dim = capsule_dim
        self.fc1 = nn.Linear(in_features, 256)
        self.fc2 = nn.Linear(256, num_classes * capsule_dim)
        self.bn  = nn.BatchNorm1d(num_classes)

    def forward(self, x):
        # x: (B, in_features)
        B = x.size(0)
        h = F.gelu(self.fc1(x))
        h = self.fc2(h).view(B, self.num_classes, self.capsule_dim)   # (B, K, cap)

        # x repeated for each class, then sliced to capsule_dim
        x_exp = x.unsqueeze(1).expand(B, self.num_classes, -1)        # (B, K, feat)
        x_sliced = x_exp[..., :self.capsule_dim]                       # (B, K, cap)

        caps = (x_sliced * h).sum(dim=-1)                              # (B, K)
        caps = self.bn(caps)
        return caps


class StrokeTopologyModule(nn.Module):
    """
    Multi-scale feature extraction tuned for Thai script strokes.
    Three parallel 5×5 convolutions (standard, dilated-1, dilated-2).
    """
    def __init__(self, in_channels: int, out_features: int):
        super().__init__()
        self.loop    = nn.Conv2d(in_channels, 64, 5, padding=2, bias=False)
        self.fine    = nn.Conv2d(in_channels, 32, 5, padding=2,
                                 dilation=1, bias=False)
        self.context = nn.Conv2d(in_channels, 32, 5, padding=4,
                                 dilation=2, bias=False)
        self.bn      = nn.BatchNorm2d(128)
        self.fc      = nn.Linear(128, out_features)

    def forward(self, x):
        loop    = F.gelu(self.loop(x))
        fine    = F.gelu(self.fine(x))
        context = F.gelu(self.context(x))
        out = torch.cat([loop, fine, context], dim=1)   # (B,128,H,W)
        out = self.bn(out)
        out = out.mean(dim=[2, 3])                       # GAP → (B,128)
        out = F.gelu(self.fc(out))                       # (B, out_features)
        return out


class CrossScaleTransformerBridge(nn.Module):
    """
    Multi-head self-attention over tokens extracted from three encoder scales.
    """
    def __init__(self, in_channels_list: list, dim: int = 256, num_heads: int = 4):
        super().__init__()
        self.projs = nn.ModuleList([
            nn.Linear(c, dim) for c in in_channels_list
        ])
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.ln   = nn.LayerNorm(dim)
        self.fc   = nn.Linear(dim, dim)

    def forward(self, *scale_maps):
        # Each scale_map: (B, C, H, W) → GAP → (B, C) → projected (B, dim)
        tokens = []
        for proj, fmap in zip(self.projs, scale_maps):
            gap = fmap.mean(dim=[2, 3])       # (B, C)
            tok = F.gelu(proj(gap))           # (B, dim)
            tokens.append(tok.unsqueeze(1))   # (B, 1, dim)

        seq = torch.cat(tokens, dim=1)                   # (B, 3, dim)
        attn_out, _ = self.attn(seq, seq, seq)
        attn_out = self.ln(attn_out + seq)
        pooled   = attn_out.mean(dim=1)                  # (B, dim)
        pooled   = F.gelu(self.fc(pooled))
        return pooled


# ─────────────────────────────────────────────────────────────────────────────
#  5. MODEL DEFINITION
# ─────────────────────────────────────────────────────────────────────────────
class WhatNetThai(nn.Module):
    def __init__(self, num_classes: int, image_size: int = 64):
        super().__init__()
        K = num_classes

        # ── Stem: triple-path (3×3, 1×5, 5×1) ────────────────────────────
        self.stem_t  = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32))
        self.stem_h  = nn.Sequential(
            nn.Conv2d(1, 32, (1, 5), padding=(0, 2), bias=False), nn.BatchNorm2d(32))
        self.stem_v  = nn.Sequential(
            nn.Conv2d(1, 32, (5, 1), padding=(2, 0), bias=False), nn.BatchNorm2d(32))

        self.stem_ca = ChannelAttention(96, reduction=8)

        self.stem_pw = nn.Sequential(
            nn.Conv2d(96, 96, 1, bias=False), nn.BatchNorm2d(96))

        # ── Encoder ────────────────────────────────────────────────────────
        self.enc1 = DenseResBlock(96,  96)    # 64→32
        self.enc2 = DenseResBlock(96,  192)   # 32→16
        self.enc3 = DenseResBlock(192, 384)   # 16→8

        # ── Cross-scale transformer bridge ─────────────────────────────────
        self.cstb = CrossScaleTransformerBridge([96, 192, 384], dim=256, num_heads=4)

        # ── AFC ─────────────────────────────────────────────────────────────
        self.afc_proj = nn.Linear(384, 256, bias=False)
        self.afc      = AdaptiveFilterCapsule(256, K, capsule_dim=8)

        # ── Stroke Topology Module ──────────────────────────────────────────
        self.stgm = StrokeTopologyModule(384, 256)

        # ── Gating ─────────────────────────────────────────────────────────
        self.gate = nn.Linear(256 + K, 2)    # combined → 2-way soft gate

        # ── Final classifier ────────────────────────────────────────────────
        self.head = nn.Sequential(
            nn.Linear(256 + K, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Linear(512, K),
        )

    def forward(self, x):
        # Stem
        t    = F.gelu(self.stem_t(x))
        h    = F.gelu(self.stem_h(x))
        v    = F.gelu(self.stem_v(x))
        stem = torch.cat([t, h, v], dim=1)        # (B, 96, H, W)
        stem = self.stem_ca(stem)
        stem = F.gelu(self.stem_pw(stem))

        # Encoder
        enc1 = self.enc1(stem)   # (B, 96,  32, 32)
        enc2 = self.enc2(enc1)   # (B, 192, 16, 16)
        enc3 = self.enc3(enc2)   # (B, 384,  8,  8)

        # CSTB
        cstb_out = self.cstb(enc1, enc2, enc3)     # (B, 256)

        # AFC
        afc_in  = enc3.mean(dim=[2, 3])            # GAP (B, 384)
        afc_in  = F.gelu(self.afc_proj(afc_in))   # (B, 256)
        afc_in  = afc_in + cstb_out
        afc_out = self.afc(afc_in)                 # (B, K)

        # STGM
        stgm_out = self.stgm(enc3)                 # (B, 256)

        # Gate
        combined = torch.cat([stgm_out, afc_out], dim=1)   # (B, 256+K)
        gate     = F.softmax(self.gate(combined), dim=-1)   # (B, 2)
        stgm_scaled = stgm_out * gate[:, 0:1]               # (B, 256)

        # Fuse & classify
        fused = torch.cat([stgm_scaled, afc_out], dim=1)    # (B, 256+K)
        out   = self.head(fused)                             # (B, K)
        return out


# ─────────────────────────────────────────────────────────────────────────────
#  6. LR SCHEDULE  (cosine annealing)
# ─────────────────────────────────────────────────────────────────────────────
class CosineAnnealingLR(torch.optim.lr_scheduler.LambdaLR):
    """Cosine annealing from base_lr to ~0 over total_steps."""
    def __init__(self, optimizer, total_steps: int, base_lr: float, min_lr: float = 1e-6):
        self.total_steps = total_steps
        self.min_lr      = min_lr
        self.base_lr     = base_lr

        def lr_lambda(step):
            cosine = 0.5 * (1.0 + math.cos(math.pi * step / max(1, total_steps)))
            target = base_lr * cosine
            return max(target, min_lr) / base_lr

        super().__init__(optimizer, lr_lambda=lr_lambda)

# ─────────────────────────────────────────────────────────────────────────────
#  7. TRAINING & EVALUATION HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * imgs.size(0)
        preds       = logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += imgs.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss   = criterion(logits, labels)

        total_loss += loss.item() * imgs.size(0)
        preds       = logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += imgs.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def compute_macro_f1(model, loader, num_classes, device) -> float:
    model.eval()
    tp = np.zeros(num_classes)
    fp = np.zeros(num_classes)
    fn = np.zeros(num_classes)

    for imgs, labels in loader:
        imgs = imgs.to(device)
        preds = model(imgs).argmax(dim=1).cpu().numpy()
        lbls  = labels.numpy()
        for c in range(num_classes):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))

    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

# ─────────────────────────────────────────────────────────────────────────────
#  8. MAIN TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────
steps_per_epoch = len(train_loader)
total_steps     = steps_per_epoch * CFG["epochs"]

MODELS = {
    "WhatNet-Thai": WhatNetThai(NUM_CLASSES, IMG),
}

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting benchmark: {len(MODELS)} model(s) × {CFG['epochs']} epochs"
         f"  [Thai folder dataset, {NUM_CLASSES} classes]", "bold"))
print(_c(f"{'═'*70}\n", "cyan"))

trained_models = {}
all_histories  = {}
results        = {}

for name, model in MODELS.items():
    model = model.to(DEVICE)
    print_model_summary(model, name)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CFG["lr"],
        weight_decay=CFG["weight_decay"],
    )
    scheduler = CosineAnnealingLR(optimizer, total_steps, CFG["lr"])
    criterion = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])

    ckpt_path  = os.path.join(CFG["results_dir"], f"{name}_best.pt")
    best_val   = 0.0
    history    = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}

    print(f"\n{_c('  ▶ Training:', 'bold', 'cyan')} {_c(name, 'yellow')}")
    BAR_W = 20
    t0    = time.time()
    patience_counter = 0
    PATIENCE = 10

    for epoch in range(CFG["epochs"]):
        ep_start = time.time()

        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, scheduler, criterion, DEVICE)
        val_loss, val_acc = evaluate(
            model, val_loader, criterion, DEVICE)

        history["loss"].append(train_loss)
        history["accuracy"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_acc)

        # ── Save best checkpoint ──
        if val_acc > best_val:
            best_val = val_acc
            torch.save(model.state_dict(), ckpt_path)
            patience_counter = 0
        else:
            patience_counter += 1

        elapsed   = time.time() - ep_start
        ep_num    = epoch + 1
        pct       = ep_num / CFG["epochs"]
        filled    = int(BAR_W * pct)
        bar       = "█" * filled + "░" * (BAR_W - filled)
        current_lr = scheduler.get_last_lr()[0]

        print(
            f"  {_c(f'Epoch {ep_num:>3}/{CFG[\"epochs\"]}', 'grey')}  "
            f"{_c(bar, 'cyan')} {_c(f'{pct*100:>5.1f}%', 'yellow')}  "
            f"{_c(f'loss={train_loss:.4f}', 'white')}  "
            f"{_c(f'acc={train_acc*100:.2f}%', 'green')}  "
            f"{_c(f'val_loss={val_loss:.4f}', 'white')}  "
            f"{_c(f'val_acc={val_acc*100:.2f}%', 'yellow' if val_acc < train_acc else 'green')}  "
            f"{_c(f'lr={current_lr:.2e}', 'blue')}  {_c(f'[{elapsed:.1f}s]', 'grey')}"
        )

        # ── Early stopping ──
        if patience_counter >= PATIENCE:
            print(_c(f"\n  Early stopping at epoch {ep_num} (patience={PATIENCE}).", "yellow"))
            break

    elapsed = time.time() - t0
    print(
        f"\n  {_c('✔ Done:', 'green', 'bold')} "
        f"best val acc = {_c(f'{best_val*100:.2f}%', 'green')}  "
        f"wall time = {_c(f'{elapsed:.0f}s', 'grey')}"
    )

    # ── Restore best weights ──
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    trained_models[name] = model
    all_histories[name]  = history

# ─────────────────────────────────────────────────────────────────────────────
#  9. FINAL TEST-SET EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
criterion_plain = nn.CrossEntropyLoss()

for name, model in trained_models.items():
    test_loss, test_acc_raw = evaluate(model, test_loader, criterion_plain, DEVICE)
    test_acc  = test_acc_raw * 100.0
    macro_f1  = compute_macro_f1(model, test_loader, NUM_CLASSES, DEVICE)
    trainable, _ = count_params(model)
    results[name] = {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    trainable,
        "test_loss": round(float(test_loss), 4),
    }

print_comparison_table(results)

# ─────────────────────────────────────────────────────────────────────────────
#  10. PERSIST RESULTS
# ─────────────────────────────────────────────────────────────────────────────
results_path = os.path.join(CFG["results_dir"], "thai_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results   → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "thai_histories.json")
with open(histories_path, "w") as f:
    json.dump(
        {n: {k: [float(v) for v in vals] for k, vals in h.items()}
         for n, h in all_histories.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] Thai folder-dataset benchmark complete.\n", "green", "bold"))

In [ ]:
[INFO] Using device: cuda
[INFO] Using CFG num_classes = 68.
[WARN] train has 6 class folders but test has 68 – labels may be inconsistent.
[WARN] Folder count (6) != NUM_CLASSES (68). Updating NUM_CLASSES.
[INFO] Classes (first 5): ['05-166-A6-KHO RAKHANG', '18-179-B3-NO NEN', '34-195-C3-RO RUA', '37-198-C6-LU', '49-210-D2-SARA AA'] …
[INFO] Indexing train images …
[INFO] Train: 4,404 | Val: 489 | Test: 1,200
[96m
══════════════════════════════════════════════════════════════════════[0m
[1m  Starting benchmark: 1 model(s) × 100 epochs  [Thai folder dataset, 6 classes][0m
[96m══════════════════════════════════════════════════════════════════════
[0m

[96m╔══════════════════════════════════════════════════════════════╗[0m
[96m[1m║  WhatNet-Thai  –  Parameter Summary                          ║[0m
[96m╠══════════════════╦═══════════════════════╦══════════════════╣[0m
[96m[1m║  Layer           ║  Type                 ║           Params  ║[0m
[96m╠══════════════════╬═══════════════════════╬══════════════════╣[0m
║  stem_t.0        ║  Conv2d               ║              288  ║
║  stem_t.1        ║  BatchNorm2d          ║               64  ║
║  stem_h.0        ║  Conv2d               ║              160  ║
║  stem_h.1        ║  BatchNorm2d          ║               64  ║
║  stem_v.0        ║  Conv2d               ║              160  ║
║  stem_v.1        ║  BatchNorm2d          ║               64  ║
║  stem_ca.fc1     ║  Linear               ║            1,164  ║
║  stem_ca.fc2     ║  Linear               ║            1,248  ║
║  stem_pw.0       ║  Conv2d               ║            9,216  ║
║  stem_pw.1       ║  BatchNorm2d          ║              192  ║
║  enc1.r1.conv1   ║  Conv2d               ║           82,944  ║
║  enc1.r1.bn1     ║  BatchNorm2d          ║              192  ║
║  enc1.r1.conv2   ║  Conv2d               ║           82,944  ║
║  enc1.r1.bn2     ║  BatchNorm2d          ║              192  ║
║  enc1.r2.conv1   ║  Conv2d               ║           82,944  ║
║  enc1.r2.bn1     ║  BatchNorm2d          ║              192  ║
║  enc1.r2.conv2   ║  Conv2d               ║           82,944  ║
║  enc1.r2.bn2     ║  BatchNorm2d          ║              192  ║
║  enc1.r3.conv1   ║  Conv2d               ║           82,944  ║
║  enc1.r3.bn1     ║  BatchNorm2d          ║              192  ║
║  … (truncated)   ║                       ║                   ║
[96m╠══════════════════╩═══════════════════════╩══════════════════╣[0m
[92m║  Trainable params                      :         13,427,900  ║[0m
[90m║  Non-trainable params                  :                  0  ║[0m
[1m[97m║  Total params                          :         13,427,900  ║[0m
[96m╚══════════════════════════════════════════════════════════════╝[0m

[1m[96m  ▶ Training:[0m [93mWhatNet-Thai[0m
[90mEpoch   1/100[0m  [96m░░░░░░░░░░░░░░░░░░░░[0m [93m  1.0%[0m  [97mloss=1.7672[0m  [92macc=18.86%[0m  [97mval_loss=1.7365[0m  [92mval_acc=19.43%[0m  [94mlr=5.00e-04[0m  [90m[3.0s][0m
[90mEpoch   2/100[0m  [96m░░░░░░░░░░░░░░░░░░░░[0m [93m  2.0%[0m  [97mloss=1.6534[0m  [92macc=27.05%[0m  [97mval_loss=1.3353[0m  [92mval_acc=47.03%[0m  [94mlr=5.00e-04[0m  [90m[3.0s][0m
[90mEpoch   3/100[0m  [96m░░░░░░░░░░░░░░░░░░░░[0m [93m  3.0%[0m  [97mloss=0.7739[0m  [92macc=82.03%[0m  [97mval_loss=0.8020[0m  [92mval_acc=82.41%[0m  [94mlr=4.99e-04[0m  [90m[3.0s][0m
[90mEpoch   4/100[0m  [96m░░░░░░░░░░░░░░░░░░░░[0m [93m  4.0%[0m  [97mloss=0.5353[0m  [92macc=95.06%[0m  [97mval_loss=0.5300[0m  [93mval_acc=94.48%[0m  [94mlr=4.98e-04[0m  [90m[3.0s][0m
[90mEpoch   5/100[0m  [96m█░░░░░░░░░░░░░░░░░░░[0m [93m  5.0%[0m  [97mloss=0.4755[0m  [92macc=97.77%[0m  [97mval_loss=0.7866[0m  [93mval_acc=82.00%[0m  [94mlr=4.97e-04[0m  [90m[2.9s][0m
[90mEpoch   6/100[0m  [96m█░░░░░░░░░░░░░░░░░░░[0m [93m  6.0%[0m  [97mloss=0.4728[0m  [92macc=97.77%[0m  [97mval_loss=0.5503[0m  [93mval_acc=94.68%[0m  [94mlr=4.96e-04[0m  [90m[3.0s][0m
[90mEpoch   7/100[0m  [96m█░░░░░░░░░░░░░░░░░░░[0m [93m  7.0%[0m  [97mloss=0.4582[0m  [92macc=98.60%[0m  [97mval_loss=0.4615[0m  [93mval_acc=98.36%[0m  [94mlr=4.94e-04[0m  [90m[3.0s][0m
[90mEpoch   8/100[0m  [96m█░░░░░░░░░░░░░░░░░░░[0m [93m  8.0%[0m  [97mloss=0.4509[0m  [92macc=98.92%[0m  [97mval_loss=0.4862[0m  [93mval_acc=97.14%[0m  [94mlr=4.92e-04[0m  [90m[2.9s][0m
[90mEpoch   9/100[0m  [96m█░░░░░░░░░░░░░░░░░░░[0m [93m  9.0%[0m  [97mloss=0.4501[0m  [92macc=98.81%[0m  [97mval_loss=0.4342[0m  [92mval_acc=99.39%[0m  [94mlr=4.90e-04[0m  [90m[3.0s][0m
[90mEpoch  10/100[0m  [96m██░░░░░░░░░░░░░░░░░░[0m [93m 10.0%[0m  [97mloss=0.4512[0m  [92macc=98.76%[0m  [97mval_loss=0.4712[0m  [93mval_acc=97.96%[0m  [94mlr=4.88e-04[0m  [90m[2.9s][0m
[90mEpoch  11/100[0m  [96m██░░░░░░░░░░░░░░░░░░[0m [93m 11.0%[0m  [97mloss=0.4453[0m  [92macc=98.97%[0m  [97mval_loss=0.5072[0m  [93mval_acc=96.73%[0m  [94mlr=4.85e-04[0m  [90m[2.9s][0m
[90mEpoch  12/100[0m  [96m██░░░░░░░░░░░░░░░░░░[0m [93m 12.0%[0m  [97mloss=0.4398[0m  [92macc=99.36%[0m  [97mval_loss=0.4334[0m  [92mval_acc=99.59%[0m  [94mlr=4.82e-04[0m  [90m[3.0s][0m
[90mEpoch  13/100[0m  [96m██░░░░░░░░░░░░░░░░░░[0m [93m 13.0%[0m  [97mloss=0.4361[0m  [92macc=99.33%[0m  [97mval_loss=0.5346[0m  [93mval_acc=95.30%[0m  [94mlr=4.79e-04[0m  [90m[2.9s][0m
[90mEpoch  14/100[0m  [96m██░░░░░░░░░░░░░░░░░░[0m [93m 14.0%[0m  [97mloss=0.4466[0m  [92macc=98.99%[0m  [97mval_loss=0.4319[0m  [92mval_acc=100.00%[0m  [94mlr=4.76e-04[0m  [90m[3.0s][0m
[90mEpoch  15/100[0m  [96m███░░░░░░░░░░░░░░░░░[0m [93m 15.0%[0m  [97mloss=0.4393[0m  [92macc=99.33%[0m  [97mval_loss=0.4499[0m  [93mval_acc=98.98%[0m  [94mlr=4.73e-04[0m  [90m[2.9s][0m
[90mEpoch  16/100[0m  [96m███░░░░░░░░░░░░░░░░░[0m [93m 16.0%[0m  [97mloss=0.4320[0m  [92macc=99.59%[0m  [97mval_loss=0.4483[0m  [93mval_acc=99.18%[0m  [94mlr=4.69e-04[0m  [90m[2.9s][0m
[90mEpoch  17/100[0m  [96m███░░░░░░░░░░░░░░░░░[0m [93m 17.0%[0m  [97mloss=0.4332[0m  [92macc=99.49%[0m  [97mval_loss=0.4414[0m  [93mval_acc=99.39%[0m  [94mlr=4.65e-04[0m  [90m[2.9s][0m
[90mEpoch  18/100[0m  [96m███░░░░░░░░░░░░░░░░░[0m [93m 18.0%[0m  [97mloss=0.4316[0m  [92macc=99.54%[0m  [97mval_loss=0.5123[0m  [93mval_acc=95.91%[0m  [94mlr=4.61e-04[0m  [90m[2.9s][0m
[90mEpoch  19/100[0m  [96m███░░░░░░░░░░░░░░░░░[0m [93m 19.0%[0m  [97mloss=0.4409[0m  [92macc=99.08%[0m  [97mval_loss=0.4425[0m  [92mval_acc=99.18%[0m  [94mlr=4.57e-04[0m  [90m[2.9s][0m
[90mEpoch  20/100[0m  [96m████░░░░░░░░░░░░░░░░[0m [93m 20.0%[0m  [97mloss=0.4452[0m  [92macc=98.97%[0m  [97mval_loss=0.6486[0m  [93mval_acc=89.16%[0m  [94mlr=4.52e-04[0m  [90m[2.9s][0m
[90mEpoch  21/100[0m  [96m████░░░░░░░░░░░░░░░░[0m [93m 21.0%[0m  [97mloss=0.4372[0m  [92macc=99.31%[0m  [97mval_loss=0.4308[0m  [92mval_acc=99.59%[0m  [94mlr=4.48e-04[0m  [90m[2.9s][0m
[90mEpoch  22/100[0m  [96m████░░░░░░░░░░░░░░░░[0m [93m 22.0%[0m  [97mloss=0.4380[0m  [92macc=99.29%[0m  [97mval_loss=0.4406[0m  [93mval_acc=98.98%[0m  [94mlr=4.43e-04[0m  [90m[2.9s][0m
[90mEpoch  23/100[0m  [96m████░░░░░░░░░░░░░░░░[0m [93m 23.0%[0m  [97mloss=0.4345[0m  [92macc=99.45%[0m  [97mval_loss=0.4385[0m  [93mval_acc=98.98%[0m  [94mlr=4.38e-04[0m  [90m[2.9s][0m
[90mEpoch  24/100[0m  [96m████░░░░░░░░░░░░░░░░[0m [93m 24.0%[0m  [97mloss=0.4337[0m  [92macc=99.54%[0m  [97mval_loss=0.4273[0m  [92mval_acc=99.80%[0m  [94mlr=4.32e-04[0m  [90m[2.9s][0m
[93m
  Early stopping at epoch 24 (patience=10).[0m

  [92m[1m✔ Done:[0m best val acc = [92m100.00%[0m  wall time = [90m71s[0m

[96m[1m╔══════════════════════════════════════════════════════════════════════╗[0m
[96m[1m║  FINAL TEST-SET COMPARISON                                           ║[0m
[96m╠════════════════════════╦════════════╦════════════╦════════════╦══════╣[0m
[96m[1m║  Model                 ║     Params ║   Test Acc ║   Macro F1 ║ Loss ║[0m
[96m╠════════════════════════╬════════════╬════════════╬════════════╬══════╣[0m
[92m[1m║★ WhatNet-Thai          ║13,427,900 ║     99.83%║     99.83%║0.099 ║[0m
[96m╚════════════════════════╩════════════╩════════════╩════════════╩══════╝[0m
[92m[1m
  ★  Winner: WhatNet-Thai  (99.83% test accuracy)
[0m
[INFO] Results   → ./results/thai_results.json
[INFO] Histories → ./results/thai_histories.json
[92m[1m
[DONE] Thai folder-dataset benchmark complete.
[0m

# Method 2

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  WhatNet  –  Thai Script Recognition  (folder-based dataset)
#
#  Dataset layout expected:
#    thai-dataset/
#      train/
#        00-161-A1-KO KAI/        ← folder name = class label
#          img001.png
#          img002.png
#          ...
#        01-162-A2-KHO KHAI/
#          ...
#      test/
#        00-161-A1-KO KAI/
#          ...
#
#  Labels are derived from the folder sort order (0-indexed),
#  so train and test folders must have identical class ordering.
# ─────────────────────────────────────────────────────────────────────────────
import os, time, random, json
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# ─────────────────────────────────────────────────────────────────────────────
#  0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  1. CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    # Will be auto-detected from the number of class folders found on disk.
    # Override here only if you want to force a specific value.
    "num_classes":      68,       # None = auto-detect
    "image_size":       64,
    "batch_size":       64,
    "epochs":           100,
    "lr":               5e-4,
    "weight_decay":     1e-4,
    "label_smoothing":  0.1,
    "val_split":        0.1,

    # Root folder that contains "train/" and "test/" sub-directories.
    # Kaggle path example: "/kaggle/input/thai-dataset"
    "data_dir":    "/kaggle/input/datasets/rautranjit/thai-dataset",

    "results_dir": "./results",
}
os.makedirs(CFG["results_dir"], exist_ok=True)

IMG     = CFG["image_size"]
BS      = CFG["batch_size"]
AUTOTUNE = tf.data.AUTOTUNE

# ─────────────────────────────────────────────────────────────────────────────
#  2. DATASET PIPELINE  (folder-image loader)
# ─────────────────────────────────────────────────────────────────────────────
train_dir = os.path.join(CFG["data_dir"], "train")
test_dir  = os.path.join(CFG["data_dir"], "test")

for d in (train_dir, test_dir):
    if not os.path.isdir(d):
        raise FileNotFoundError(
            f"[ERROR] Expected directory not found: {d}\n"
            f"Make sure CFG['data_dir'] points to the folder that contains "
            f"'train/' and 'test/' sub-directories."
        )

# ── Auto-detect num_classes from folder count ─────────────────────────────────
_train_classes = sorted([
    c for c in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, c))
])
_test_classes  = sorted([
    c for c in os.listdir(test_dir)
    if os.path.isdir(os.path.join(test_dir, c))
])

if CFG["num_classes"] is None:
    NUM_CLASSES = len(_train_classes)
    print(f"[INFO] Auto-detected {NUM_CLASSES} classes from train folder.")
else:
    NUM_CLASSES = CFG["num_classes"]
    print(f"[INFO] Using CFG num_classes = {NUM_CLASSES}.")

if len(_train_classes) != len(_test_classes):
    print(f"[WARN] train has {len(_train_classes)} class folders but "
          f"test has {len(_test_classes)} – labels may be inconsistent.")

if len(_train_classes) != NUM_CLASSES:
    print(f"[WARN] Folder count ({len(_train_classes)}) != NUM_CLASSES "
          f"({NUM_CLASSES}).  Updating NUM_CLASSES.")
    NUM_CLASSES = len(_train_classes)

print(f"[INFO] Classes (first 5): {_train_classes[:5]} …")

# ── Helper: load a split from an image folder ────────────────────────────────
def load_folder_split(root: str, class_names: list) -> tuple:
    """
    Walk root/<class_name>/* and return (images_uint8, labels_int32).

    Images are resized to (IMG, IMG) and converted to greyscale.
    Labels are the sorted folder index (0-based).
    """
    class_to_idx = {c: i for i, c in enumerate(class_names)}
    images, labels = [], []

    for cls in class_names:
        cls_dir = os.path.join(root, cls)
        if not os.path.isdir(cls_dir):
            print(f"[WARN] Missing class folder: {cls_dir}")
            continue
        idx = class_to_idx[cls]
        for fname in sorted(os.listdir(cls_dir)):
            fpath = os.path.join(cls_dir, fname)
            if not os.path.isfile(fpath):
                continue
            ext = os.path.splitext(fname)[1].lower()
            if ext not in {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}:
                continue
            try:
                raw    = tf.io.read_file(fpath)
                # decode_image handles PNG / JPEG / BMP / GIF automatically
                img    = tf.image.decode_image(raw, channels=1, expand_animations=False)
                img    = tf.image.resize(img, [IMG, IMG])
                img    = tf.cast(img, tf.uint8)
                images.append(img.numpy())
                labels.append(idx)
            except Exception as e:
                print(f"[WARN] Could not read {fpath}: {e}")

    images_np = np.array(images, dtype=np.uint8)   # (N, IMG, IMG, 1)
    labels_np = np.array(labels, dtype=np.int32)
    return images_np, labels_np

print("[INFO] Loading train images …  (this may take a moment)")
x_train_all, y_train_all = load_folder_split(train_dir, _train_classes)
print(f"[INFO] Train loaded: {len(x_train_all):,} images")

print("[INFO] Loading test images …")
x_test, y_test = load_folder_split(test_dir, _train_classes)
print(f"[INFO] Test  loaded: {len(x_test):,} images")

# ── Squeeze channel dim for val split (re-added later) ───────────────────────
# images are currently (N, IMG, IMG, 1)  – squeeze to (N, IMG, IMG) for split
x_train_all = x_train_all[..., 0]   # → (N, IMG, IMG)
x_test      = x_test[..., 0]         # → (N, IMG, IMG)

# ── Train / val split ─────────────────────────────────────────────────────────
n_total = len(x_train_all)
n_val   = max(1, int(n_total * CFG["val_split"]))
n_train = n_total - n_val

rng  = np.random.default_rng(SEED)
perm = rng.permutation(n_total)
x_train_all, y_train_all = x_train_all[perm], y_train_all[perm]

x_train, y_train = x_train_all[:n_train], y_train_all[:n_train]
x_val,   y_val   = x_train_all[n_train:], y_train_all[n_train:]

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {len(x_test):,}")

# ── Add channel dim: (N,IMG,IMG) → (N,IMG,IMG,1) ─────────────────────────────
x_train = x_train[..., np.newaxis]
x_val   = x_val[...,   np.newaxis]
x_test  = x_test[...,  np.newaxis]

# ── Rotation augmentation ─────────────────────────────────────────────────────
_HAS_RANDOM_ROTATION = hasattr(tf.keras.layers, "RandomRotation")
if _HAS_RANDOM_ROTATION:
    print("[INFO] RandomRotation available – rotation augmentation enabled (±8°).")
    _rotator = tf.keras.layers.RandomRotation(
        factor=8/360,
        fill_mode="constant",
        fill_value=-1.0,
        seed=SEED,
    )
else:
    print("[WARN] RandomRotation not available – rotation augmentation disabled.")
    _rotator = None

# ── Preprocessing helpers ─────────────────────────────────────────────────────
def normalise(img, lbl):
    img = tf.cast(img, tf.float32) / 127.5 - 1.0
    return img, lbl

def augment(img, lbl):
    """
    Stochastic augmentation – training only.

    Thai consonants have:
      • Circular loops that sit at varying heights/positions
      • Diacritics (sara) that hover above or below the baseline
      • Significant pen-angle variation in handwriting

    Augmentations:
      • Brightness / contrast jitter – ink variation
      • Pad-then-random-crop         – translation ±4 px (64-px image)
      • Random rotation ±8°          – writing angle variation

    No flip: Thai script is strongly directional.
    """
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    # ±4 px translation on 64-px images (pad=4 each side → crop back to 64)
    img = tf.pad(img, [[4, 4], [4, 4], [0, 0]], constant_values=-1.0)
    img = tf.image.random_crop(img, [IMG, IMG, 1])
    if _rotator is not None:
        img = _rotator(tf.expand_dims(img, 0))[0]
    return img, lbl

def to_onehot(img, lbl):
    return img, tf.one_hot(lbl, NUM_CLASSES)

# ── Build tf.data pipelines ───────────────────────────────────────────────────
train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .map(augment,   num_parallel_calls=AUTOTUNE)
    .map(to_onehot, num_parallel_calls=AUTOTUNE)
    .shuffle(8192, seed=SEED)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .map(to_onehot, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds_oh = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)

# ─────────────────────────────────────────────────────────────────────────────
#  3. DISPLAY UTILITIES
# ─────────────────────────────────────────────────────────────────────────────
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"

def print_model_summary(model: Model) -> None:
    W             = 62
    trainable     = model.count_params()
    non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
    total         = trainable + non_trainable
    title         = f"  {model.name}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╦{'═'*23}╦{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Layer':<16}║  {'Type':<21}║  {'Params':>15}  ║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╬{'═'*23}╬{'═'*18}╣", "cyan"))
    for lyr in [l for l in model.layers if l.count_params() > 0][:20]:
        n_p = lyr.count_params()
        print(f"║  {lyr.name[:14]:<16}║  {type(lyr).__name__[:21]:<21}║  {n_p:>15,}  ║")
    if len([l for l in model.layers if l.count_params() > 0]) > 20:
        print(f"║  {'… (truncated)':<16}║  {'':21}║  {'':>15}  ║")
    print(_c(f"╠{'═'*18}╩{'═'*23}╩{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {non_trainable:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))

class EpochProgressCallback(keras.callbacks.Callback):
    BAR_WIDTH = 20
    def __init__(self, total_epochs: int, model_name: str):
        super().__init__()
        self.total_epochs = total_epochs
        self.model_name   = model_name
        self._epoch_start = 0.0

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        logs    = logs or {}
        elapsed = time.time() - self._epoch_start
        ep_num  = epoch + 1
        pct     = ep_num / self.total_epochs
        filled  = int(self.BAR_WIDTH * pct)
        bar     = "█" * filled + "░" * (self.BAR_WIDTH - filled)
        loss    = logs.get("loss",         float("nan"))
        acc     = logs.get("accuracy",     float("nan")) * 100
        val_acc = logs.get("val_accuracy", float("nan")) * 100
        val_los = logs.get("val_loss",     float("nan"))
        try:
            lr_val = float(keras.backend.get_value(self.model.optimizer.learning_rate))
            lr_str = f"lr={lr_val:.2e}"
        except Exception:
            lr_str = ""
        print(
            f"  {_c(f'Epoch {ep_num:>3}/{self.total_epochs}', 'grey')}  "
            f"{_c(bar, 'cyan')} {_c(f'{pct*100:>5.1f}%', 'yellow')}  "
            f"{_c(f'loss={loss:.4f}', 'white')}  {_c(f'acc={acc:.2f}%', 'green')}  "
            f"{_c(f'val_loss={val_los:.4f}', 'white')}  "
            f"{_c(f'val_acc={val_acc:.2f}%', 'yellow' if val_acc < acc else 'green')}  "
            f"{_c(lr_str, 'blue')}  {_c(f'[{elapsed:.1f}s]', 'grey')}"
        )

def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║",
             "cyan", "bold"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n",
             "green", "bold"))

# ─────────────────────────────────────────────────────────────────────────────
#  4. BUILDING BLOCKS
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x):
    return tf.nn.gelu(x)

def residual_block(x, channels: int):
    residual = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, residual])
    x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_channels: int, out_channels: int):
    if in_channels != out_channels:
        skip = layers.Conv2D(out_channels, 1, use_bias=False)(x)
        skip = layers.BatchNormalization()(skip)
        x_in = layers.Activation(gelu)(skip)
    else:
        x_in = x
    r1  = residual_block(x_in, out_channels)
    r2  = residual_block(r1,   out_channels)
    r3  = residual_block(r2,   out_channels)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_channels, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_channels, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    return out

def channel_attention(x, channels: int, reduction: int = 8):
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(channels // reduction, activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def adaptive_filter_capsule(x, num_classes: int, capsule_dim: int = 8):
    h        = layers.Dense(256, activation=gelu)(x)
    h        = layers.Dense(num_classes * capsule_dim)(h)
    h        = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps     = layers.Multiply()([x_sliced, h])
    caps     = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    caps     = layers.BatchNormalization()(caps)
    return caps

# ─────────────────────────────────────────────────────────────────────────────
#  5. MODEL DEFINITION
# ─────────────────────────────────────────────────────────────────────────────
def build_whatnet_thai(num_classes: int, image_size: int = 64, head_units: int = 512) -> Model:
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # ── Stem: triple-path (3×3, 1×5, 5×1) ────────────────────────────────
    t = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t = layers.BatchNormalization()(t)
    t = layers.Activation(gelu)(t)

    h = layers.Conv2D(32, (1, 5), padding="same", use_bias=False)(inp)
    h = layers.BatchNormalization()(h)
    h = layers.Activation(gelu)(h)

    v = layers.Conv2D(32, (5, 1), padding="same", use_bias=False)(inp)
    v = layers.BatchNormalization()(v)
    v = layers.Activation(gelu)(v)

    stem = layers.Concatenate()([t, h, v])          # 96 channels
    stem = channel_attention(stem, 96)
    stem = layers.Conv2D(96, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem)
    stem = layers.Activation(gelu)(stem)

    # ── Encoder (no scaffold bypass) ──────────────────────────────────────
    enc1 = dense_res_block(stem, 96,  96)   # → 32×32
    enc2 = dense_res_block(enc1, 96,  192)  # → 16×16
    enc3 = dense_res_block(enc2, 192, 384)  # →  8× 8

    # ── Decoder head ──────────────────────────────────────────────────────
    # Multi-scale GAP fusion
    gap1 = layers.GlobalAveragePooling2D(name="gap1")(enc1)
    gap2 = layers.GlobalAveragePooling2D(name="gap2")(enc2)
    gap3 = layers.GlobalAveragePooling2D(name="gap3")(enc3)
    fused_gap = layers.Concatenate(name="multiscale_fused")([gap1, gap2, gap3])

    # Adaptive Filter Capsule (AFC)
    # Projects the fused multi-scale vector into capsule space.
    # Each of the K capsules learns to respond to one character class.
    afc_out = adaptive_filter_capsule(fused_gap, num_classes)   # (B, K)

    # Classification head
    # Dense projection of the raw GAP features (residual path alongside AFC)
    x = layers.Dense(head_units, use_bias=False, name="head_dense")(fused_gap)
    x = layers.LayerNormalization(name="head_ln")(x)
    x = layers.Activation("gelu", name="head_act")(x)
    x = layers.Dense(num_classes, name="head_logits")(x)

    # Gated fusion: AFC scores + dense-head logits
    # A learned scalar gate (per-sample softmax over 2 weights) blends the
    # AFC capsule scores with the plain dense logits.  This lets the model
    # learn how much to trust the capsule routing vs. the direct projection.
    combined = layers.Concatenate(name="gate_input")([x, afc_out])
    gate     = layers.Dense(2, activation="softmax", name="gate")(combined)  # (B, 2)

    # gate[:,0] weights the dense head; gate[:,1] weights the AFC output
    x_scaled   = layers.Lambda(
        lambda t: t[0] * t[1][:, 0:1], name="gate_dense")([x,gate])
    afc_scaled = layers.Lambda(
        lambda t: t[0] * t[1][:, 1:2], name="gate_afc"  )([afc_out,gate])

    outputs = layers.Add(name="logits")([x_scaled, afc_scaled])

    model = keras.Model(inputs=inp, outputs=outputs, name="our_model-Net")
    return model

MODELS_TF = {
    "WhatNet-Thai": lambda: build_whatnet_thai(NUM_CLASSES, IMG),
}

# ─────────────────────────────────────────────────────────────────────────────
#  6. LR SCHEDULE
# ─────────────────────────────────────────────────────────────────────────────
class CosineAnnealing(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base: float, steps: int):
        self.base  = base
        self.steps = tf.cast(steps, tf.float32)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        cosine = 0.5 * (1.0 + tf.cos(np.pi * step / self.steps))
        return tf.maximum(self.base * cosine, 1e-6)

    def get_config(self):
        return {"base": self.base, "steps": int(self.steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  7. TRAINING & EVALUATION HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def compile_model(model: Model, steps_total: int) -> Model:
    lr_sch = CosineAnnealing(CFG["lr"], steps_total)
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=lr_sch,
            weight_decay=CFG["weight_decay"],
        ),
        loss=keras.losses.CategoricalCrossentropy(
            from_logits=True,
            label_smoothing=CFG["label_smoothing"],
        ),
        metrics=["accuracy"],
        jit_compile=True,
    )
    return model

def compute_macro_f1(model: Model, dataset) -> float:
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for images, labels in dataset:
        preds = tf.argmax(model(images, training=False), axis=1).numpy()
        lbls  = labels.numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

# ─────────────────────────────────────────────────────────────────────────────
#  8. TRAIN + EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
trained_models  = {}
all_histories   = {}
steps_per_epoch = -(-n_train // BS)
total_steps     = steps_per_epoch * CFG["epochs"]

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting benchmark: {len(MODELS_TF)} model(s) × {CFG['epochs']} epochs"
         f"  [Thai folder dataset, {NUM_CLASSES} classes]", "bold"))
print(_c(f"{'═'*70}\n", "cyan"))

for name, model_fn in MODELS_TF.items():
    model = model_fn()
    model = compile_model(model, total_steps)
    print_model_summary(model)

    ckpt_path = os.path.join(CFG["results_dir"], f"{name}_best.keras")
    callbacks = [
        ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True, verbose=1),
        EpochProgressCallback(CFG["epochs"], name),
    ]

    print(f"\n{_c('  ▶ Training:', 'bold', 'cyan')} {_c(name, 'yellow')}")
    t0      = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CFG["epochs"],
        callbacks=callbacks,
        verbose=0,
    )
    elapsed  = time.time() - t0
    best_val = max(history.history["val_accuracy"]) * 100.0
    print(
        f"\n  {_c('✔ Done:', 'green', 'bold')} "
        f"best val acc = {_c(f'{best_val:.2f}%', 'green')}  "
        f"wall time = {_c(f'{elapsed:.0f}s', 'grey')}"
    )
    trained_models[name] = model
    all_histories[name]  = history.history

# ─────────────────────────────────────────────────────────────────────────────
#  9. FINAL TEST-SET EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
results = {}
for name, model in trained_models.items():
    test_loss, test_acc_raw = model.evaluate(test_ds_oh, verbose=0)
    test_acc  = test_acc_raw * 100.0
    macro_f1  = compute_macro_f1(model, test_ds)
    results[name] = {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    model.count_params(),
        "test_loss": round(float(test_loss), 4),
    }
print_comparison_table(results)

# ─────────────────────────────────────────────────────────────────────────────
#  10. PERSIST RESULTS
# ─────────────────────────────────────────────────────────────────────────────
results_path = os.path.join(CFG["results_dir"], "thai_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results   → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "thai_histories.json")
with open(histories_path, "w") as f:
    json.dump(
        {n: {k: [float(v) for v in vals] for k, vals in h.items()}
         for n, h in all_histories.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] Thai folder-dataset benchmark complete.\n", "green", "bold"))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  WhatNet  –  Thai Script Recognition  (PyTorch, folder-based dataset)
#
#  Dataset layout expected:
#    thai-dataset/
#      train/
#        00-161-A1-KO KAI/        ← folder name = class label
#          img001.png
#          img002.png
#          ...
#        01-162-A2-KHO KHAI/
#          ...
#      test/
#        00-161-A1-KO KAI/
#          ...
#
#  Labels are derived from the folder sort order (0-indexed),
#  so train and test folders must have identical class ordering.
# ─────────────────────────────────────────────────────────────────────────────
import os, time, random, json, math
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
import torchvision.transforms as T

# ─────────────────────────────────────────────────────────────────────────────
#  0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {DEVICE}")

# ─────────────────────────────────────────────────────────────────────────────
#  1. CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    "num_classes":      None,      # None = auto-detect from folder count
    "image_size":       64,
    "batch_size":       64,
    "epochs":           50,
    "lr":               5e-4,
    "weight_decay":     1e-4,
    "label_smoothing":  0.05,
    "val_split":        0.1,

    # Root folder that contains "train/" and "test/" sub-directories.
    "data_dir":    "/kaggle/input/datasets/rautranjit/thai-dataset",

    "results_dir": "./results",
    "num_workers": 4,
}
os.makedirs(CFG["results_dir"], exist_ok=True)

IMG = CFG["image_size"]
BS  = CFG["batch_size"]

# ─────────────────────────────────────────────────────────────────────────────
#  2. DATASET
# ─────────────────────────────────────────────────────────────────────────────
train_dir = os.path.join(CFG["data_dir"], "train")
test_dir  = os.path.join(CFG["data_dir"], "test")

for d in (train_dir, test_dir):
    if not os.path.isdir(d):
        raise FileNotFoundError(
            f"[ERROR] Expected directory not found: {d}\n"
            f"Make sure CFG['data_dir'] points to the folder that contains "
            f"'train/' and 'test/' sub-directories."
        )

_train_classes = sorted([
    c for c in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, c))
])
_test_classes = sorted([
    c for c in os.listdir(test_dir)
    if os.path.isdir(os.path.join(test_dir, c))
])

# ── Auto-detect num_classes ───────────────────────────────────────────────────
if CFG["num_classes"] is None:
    NUM_CLASSES = len(_train_classes)
    print(f"[INFO] Auto-detected {NUM_CLASSES} classes from train folder.")
else:
    NUM_CLASSES = CFG["num_classes"]
    print(f"[INFO] Using CFG num_classes = {NUM_CLASSES}.")

if len(_train_classes) != len(_test_classes):
    print(f"[WARN] train has {len(_train_classes)} class folders but "
          f"test has {len(_test_classes)} – labels may be inconsistent.")

if set(_train_classes) != set(_test_classes):
    only_train = set(_train_classes) - set(_test_classes)
    only_test  = set(_test_classes)  - set(_train_classes)
    if only_train:
        print(f"[WARN] Only in train: {sorted(only_train)}")
    if only_test:
        print(f"[WARN] Only in test : {sorted(only_test)}")

if len(_train_classes) != NUM_CLASSES:
    print(f"[WARN] Folder count ({len(_train_classes)}) != NUM_CLASSES "
          f"({NUM_CLASSES}). Updating NUM_CLASSES.")
    NUM_CLASSES = len(_train_classes)

print(f"[INFO] NUM_CLASSES = {NUM_CLASSES}")
print(f"[INFO] Classes (first 5): {_train_classes[:5]} …")

VALID_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}


class ThaiScriptDataset(Dataset):
    """
    Folder-based dataset for Thai script images.
    Loads grayscale images resized to (IMG x IMG).
    """
    def __init__(self, root: str, class_names: list, transform=None):
        self.transform    = transform
        self.class_to_idx = {c: i for i, c in enumerate(class_names)}
        self.samples      = []   # list of (path, label_int)

        for cls in class_names:
            cls_dir = os.path.join(root, cls)
            if not os.path.isdir(cls_dir):
                print(f"[WARN] Missing class folder: {cls_dir}")
                continue
            idx = self.class_to_idx[cls]
            for fname in sorted(os.listdir(cls_dir)):
                fpath = os.path.join(cls_dir, fname)
                if not os.path.isfile(fpath):
                    continue
                if os.path.splitext(fname)[1].lower() not in VALID_EXTS:
                    continue
                self.samples.append((fpath, idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        fpath, label = self.samples[index]
        try:
            img = Image.open(fpath).convert("L")           # grayscale
            img = img.resize((IMG, IMG), Image.BILINEAR)
        except Exception as e:
            print(f"[WARN] Could not read {fpath}: {e}")
            img = Image.fromarray(np.zeros((IMG, IMG), dtype=np.uint8))
        if self.transform:
            img = self.transform(img)
        return img, label


# ── Transforms  (normalise to [-1, 1], matching TF x/127.5 - 1) ──────────────
_mean = (0.5,)
_std  = (0.5,)

train_transform = T.Compose([
    T.RandomAffine(
        degrees=8,                           # ±8° rotation
        translate=(4 / IMG, 4 / IMG),        # ±4 px translation
    ),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),                            # [0, 1]
    T.Normalize(_mean, _std),               # [-1, 1]
])

eval_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(_mean, _std),
])

# ── Build datasets ─────────────────────────────────────────────────────────────
print("[INFO] Building dataset objects …")
full_train_ds = ThaiScriptDataset(train_dir, _train_classes, transform=None)
test_ds_raw   = ThaiScriptDataset(test_dir,  _train_classes, transform=None)

n_total = len(full_train_ds)
n_val   = max(1, int(n_total * CFG["val_split"]))
n_train = n_total - n_val

g = torch.Generator().manual_seed(SEED)
train_subset, val_subset = random_split(full_train_ds, [n_train, n_val], generator=g)

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {len(test_ds_raw):,}")


class TransformSubset(Dataset):
    """Wraps a Subset and applies a transform at load time."""
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        fpath, label = self.subset.dataset.samples[self.subset.indices[idx]]
        try:
            img = Image.open(fpath).convert("L").resize((IMG, IMG), Image.BILINEAR)
        except Exception:
            img = Image.fromarray(np.zeros((IMG, IMG), dtype=np.uint8))
        return self.transform(img), label


class SimpleDataset(Dataset):
    """Wraps ThaiScriptDataset and applies eval transform."""
    def __init__(self, base_ds, transform):
        self.base      = base_ds
        self.transform = transform

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        fpath, label = self.base.samples[idx]
        try:
            img = Image.open(fpath).convert("L").resize((IMG, IMG), Image.BILINEAR)
        except Exception:
            img = Image.fromarray(np.zeros((IMG, IMG), dtype=np.uint8))
        return self.transform(img), label


train_dataset = TransformSubset(train_subset, train_transform)
val_dataset   = TransformSubset(val_subset,   eval_transform)
test_dataset  = SimpleDataset(test_ds_raw,    eval_transform)


def make_loader(ds, shuffle: bool) -> DataLoader:
    return DataLoader(
        ds,
        batch_size=BS,
        shuffle=shuffle,
        num_workers=CFG["num_workers"],
        pin_memory=True,
        drop_last=False,
    )


train_loader = make_loader(train_dataset, shuffle=True)
val_loader   = make_loader(val_dataset,   shuffle=False)
test_loader  = make_loader(test_dataset,  shuffle=False)

# ─────────────────────────────────────────────────────────────────────────────
#  3. DISPLAY UTILITIES
# ─────────────────────────────────────────────────────────────────────────────
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}


def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"


def print_model_summary(model: nn.Module, name: str) -> None:
    W         = 62
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_train = total - trainable
    title     = f"  {name}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═' * W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═' * W}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {non_train:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═' * W}╝", "cyan"))


def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═' * W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║"
             f"{'Macro F1':>11} ║{'Loss':>5} ║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║"
               f"{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  "
             f"({results[best_name]['test_acc']:.2f}% test accuracy)\n",
             "green", "bold"))

# ─────────────────────────────────────────────────────────────────────────────
#  4. BUILDING BLOCKS
# ─────────────────────────────────────────────────────────────────────────────

class ResidualBlock(nn.Module):
    """Two Conv-BN-GELU layers with identity skip."""
    def __init__(self, channels: int):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)

    def forward(self, x):
        res = x
        x = F.gelu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x = F.gelu(x + res)
        return x


class DenseResBlock(nn.Module):
    """
    Three stacked ResidualBlocks whose outputs are concatenated,
    fused with a 1×1 conv, then downsampled 2× with a depthwise stride-2 conv.
    Optionally adapts the channel count via a projection shortcut.
    """
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.proj = None
        if in_channels != out_channels:
            self.proj = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, bias=False),
                nn.BatchNorm2d(out_channels),
            )

        self.r1 = ResidualBlock(out_channels)
        self.r2 = ResidualBlock(out_channels)
        self.r3 = ResidualBlock(out_channels)

        # Fuse concatenated (3×out_channels) back to out_channels
        self.fuse = nn.Sequential(
            nn.Conv2d(out_channels * 3, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

        # Depthwise stride-2 downsampling (matches TF DepthwiseConv2D strides=2)
        self.down = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, stride=2,
                      padding=1, groups=out_channels, bias=False),
            nn.Conv2d(out_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        if self.proj is not None:
            x = F.gelu(self.proj(x))

        r1  = self.r1(x)
        r2  = self.r2(r1)
        r3  = self.r3(r2)

        cat = torch.cat([r1, r2, r3], dim=1)    # (B, 3C, H, W)
        out = F.gelu(self.fuse(cat))             # (B,  C, H, W)
        out = F.gelu(self.down(out))             # (B,  C, H/2, W/2)
        return out


class ChannelAttention(nn.Module):
    """Squeeze-and-Excitation channel attention."""
    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        attn = self.fc(self.gap(x)).unsqueeze(-1).unsqueeze(-1)
        return x * attn


class AdaptiveFilterCapsule(nn.Module):
    """
    Projects the fused multi-scale vector into a K-dimensional capsule space.
    Each of the K outputs learns to respond to one character class.
    Mirrors the TF adaptive_filter_capsule() function exactly.
    """
    def __init__(self, in_features: int, num_classes: int, capsule_dim: int = 8):
        super().__init__()
        self.num_classes = num_classes
        self.capsule_dim = capsule_dim
        self.fc1 = nn.Linear(in_features, 256)
        self.fc2 = nn.Linear(256, num_classes * capsule_dim)
        self.bn  = nn.BatchNorm1d(num_classes)

    def forward(self, x):
        B    = x.size(0)
        h    = F.gelu(self.fc1(x))
        h    = self.fc2(h).view(B, self.num_classes, self.capsule_dim)  # (B, K, d)
        # RepeatVector equivalent: expand x to (B, K, F) then slice first d dims
        x_exp  = x.unsqueeze(1).expand(B, self.num_classes, -1)         # (B, K, F)
        x_sl   = x_exp[:, :, :self.capsule_dim]                         # (B, K, d)
        caps   = (x_sl * h).sum(dim=-1)                                 # (B, K)
        caps   = self.bn(caps)
        return caps

# ─────────────────────────────────────────────────────────────────────────────
#  5. MODEL DEFINITION
# ─────────────────────────────────────────────────────────────────────────────

class WhatNetThai(nn.Module):
    """
    WhatNet-Thai  –  PyTorch port of the TensorFlow build_whatnet_thai().

    Architecture:
      Stem : three parallel convolutions (3×3, 1×5, 5×1) → channel attention
      Enc1 : DenseResBlock  96 →  96   (→ 32×32)
      Enc2 : DenseResBlock  96 → 192   (→ 16×16)
      Enc3 : DenseResBlock 192 → 384   (→  8× 8)
      Head : multi-scale GAP fusion → AFC + dense head → gated logit blend
    """
    def __init__(self, num_classes: int, image_size: int = 64,
                 head_units: int = 512):
        super().__init__()
        K = num_classes

        # ── Stem ─────────────────────────────────────────────────────────────
        self.stem_t = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.GELU(),
        )
        self.stem_h = nn.Sequential(
            nn.Conv2d(1, 32, (1, 5), padding=(0, 2), bias=False),
            nn.BatchNorm2d(32), nn.GELU(),
        )
        self.stem_v = nn.Sequential(
            nn.Conv2d(1, 32, (5, 1), padding=(2, 0), bias=False),
            nn.BatchNorm2d(32), nn.GELU(),
        )
        self.stem_attn = ChannelAttention(96)
        self.stem_proj = nn.Sequential(
            nn.Conv2d(96, 96, 1, bias=False),
            nn.BatchNorm2d(96), nn.GELU(),
        )

        # ── Encoder ──────────────────────────────────────────────────────────
        self.enc1 = DenseResBlock(96,  96)    # → 32×32
        self.enc2 = DenseResBlock(96,  192)   # → 16×16
        self.enc3 = DenseResBlock(192, 384)   # →  8× 8

        # Fused feature dim = 96 + 192 + 384 = 672
        F_DIM = 96 + 192 + 384

        # ── Adaptive Filter Capsule ───────────────────────────────────────────
        self.afc = AdaptiveFilterCapsule(F_DIM, K)

        # ── Dense head ────────────────────────────────────────────────────────
        self.head_dense  = nn.Linear(F_DIM, head_units, bias=False)
        self.head_ln     = nn.LayerNorm(head_units)
        self.head_logits = nn.Linear(head_units, K)

        # ── Gated fusion ──────────────────────────────────────────────────────
        # gate[:,0] weights dense head; gate[:,1] weights AFC output
        self.gate = nn.Linear(K * 2, 2)

    def forward(self, x):
        # Stem
        t    = self.stem_t(x)
        h    = self.stem_h(x)
        v    = self.stem_v(x)
        stem = torch.cat([t, h, v], dim=1)           # (B, 96, H, W)
        stem = self.stem_attn(stem)
        stem = self.stem_proj(stem)

        # Encoder
        e1 = self.enc1(stem)                          # (B,  96, 32, 32)
        e2 = self.enc2(e1)                            # (B, 192, 16, 16)
        e3 = self.enc3(e2)                            # (B, 384,  8,  8)

        # Multi-scale GAP fusion
        gap1  = e1.mean(dim=(-2, -1))                 # (B,  96)
        gap2  = e2.mean(dim=(-2, -1))                 # (B, 192)
        gap3  = e3.mean(dim=(-2, -1))                 # (B, 384)
        fused = torch.cat([gap1, gap2, gap3], dim=1)  # (B, 672)

        # AFC branch
        afc_out = self.afc(fused)                     # (B, K)

        # Dense head branch
        x_ = F.gelu(self.head_ln(self.head_dense(fused)))
        x_ = self.head_logits(x_)                    # (B, K)

        # Gated fusion
        combined = torch.cat([x_, afc_out], dim=1)   # (B, 2K)
        gate     = F.softmax(self.gate(combined), dim=1)  # (B, 2)

        x_scaled   = x_      * gate[:, 0:1]
        afc_scaled = afc_out * gate[:, 1:2]
        logits     = x_scaled + afc_scaled            # (B, K)
        return logits

# ─────────────────────────────────────────────────────────────────────────────
#  6. LR SCHEDULE  (cosine annealing, mirrors TF CosineAnnealing)
# ─────────────────────────────────────────────────────────────────────────────

def cosine_annealing_lambda(total_steps: int, base_lr: float):
    def lr_lambda(current_step: int):
        cosine = 0.5 * (1.0 + math.cos(math.pi * current_step / total_steps))
        return max(cosine, 1e-6 / base_lr)
    return lr_lambda

# ─────────────────────────────────────────────────────────────────────────────
#  7. TRAINING & EVALUATION HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def compute_macro_f1(model: nn.Module, loader: DataLoader,
                     num_classes: int) -> float:
    model.eval()
    tp = np.zeros(num_classes)
    fp = np.zeros(num_classes)
    fn = np.zeros(num_classes)
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            preds  = model(images).argmax(dim=1).cpu().numpy()
            lbls   = labels.numpy()
            for c in range(num_classes):
                tp[c] += np.sum((preds == c) & (lbls == c))
                fp[c] += np.sum((preds == c) & (lbls != c))
                fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)


def evaluate(model: nn.Module, loader: DataLoader,
             criterion: nn.Module) -> tuple:
    """Returns (avg_loss, accuracy_pct)."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits  = model(images)
            loss    = criterion(logits, labels)
            total_loss += loss.item() * images.size(0)
            preds   = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += images.size(0)
    return total_loss / total, correct / total * 100.0

# ─────────────────────────────────────────────────────────────────────────────
#  8. TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────
steps_per_epoch = len(train_loader)
total_steps     = steps_per_epoch * CFG["epochs"]
_COL_BAR_W      = 20


def train_one_epoch(model, loader, optimizer, scheduler, criterion):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        running_loss += loss.item() * images.size(0)
        correct      += (logits.argmax(1) == labels).sum().item()
        total        += images.size(0)
    return running_loss / total, correct / total * 100.0


def train_model(model: nn.Module, name: str) -> tuple:
    model = model.to(DEVICE)
    print_model_summary(model, name)

    criterion = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])
    optimizer = AdamW(model.parameters(),
                      lr=CFG["lr"],
                      weight_decay=CFG["weight_decay"])
    scheduler = LambdaLR(optimizer,
                         cosine_annealing_lambda(total_steps, CFG["lr"]))

    best_val_acc = 0.0
    best_state   = None
    patience     = 15
    no_improve   = 0
    history      = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}

    print(f"\n{_c('  ▶ Training:', 'bold', 'cyan')} {_c(name, 'yellow')}")
    t0 = time.time()

    for epoch in range(CFG["epochs"]):
        ep_start = time.time()

        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, scheduler, criterion)
        val_loss, val_acc = evaluate(model, val_loader, criterion)

        history["loss"].append(train_loss)
        history["accuracy"].append(train_acc / 100.0)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_acc / 100.0)

        ep_num  = epoch + 1
        pct     = ep_num / CFG["epochs"]
        filled  = int(_COL_BAR_W * pct)
        bar     = "█" * filled + "░" * (_COL_BAR_W - filled)
        lr_val  = scheduler.get_last_lr()[0]
        elapsed = time.time() - ep_start
        print(
            f"  {_c(f'Epoch {ep_num:>3}/{CFG[\"epochs\"]}', 'grey')}  "
            f"{_c(bar, 'cyan')} {_c(f'{pct*100:>5.1f}%', 'yellow')}  "
            f"{_c(f'loss={train_loss:.4f}', 'white')}  "
            f"{_c(f'acc={train_acc:.2f}%', 'green')}  "
            f"{_c(f'val_loss={val_loss:.4f}', 'white')}  "
            f"{_c(f'val_acc={val_acc:.2f}%', 'yellow' if val_acc < train_acc else 'green')}  "
            f"{_c(f'lr={lr_val:.2e}', 'blue')}  {_c(f'[{elapsed:.1f}s]', 'grey')}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            ckpt_path    = os.path.join(CFG["results_dir"], f"{name}_best.pt")
            torch.save(best_state, ckpt_path)
            no_improve   = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  {_c('Early stopping triggered.', 'yellow')} "
                      f"(no val_acc improvement for {patience} epochs)")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    elapsed = time.time() - t0
    print(
        f"\n  {_c('✔ Done:', 'green', 'bold')} "
        f"best val acc = {_c(f'{best_val_acc:.2f}%', 'green')}  "
        f"wall time = {_c(f'{elapsed:.0f}s', 'grey')}"
    )
    return model, history

# ─────────────────────────────────────────────────────────────────────────────
#  9. RUN
# ─────────────────────────────────────────────────────────────────────────────
MODELS = {
    "WhatNet-Thai": lambda: WhatNetThai(NUM_CLASSES, IMG),
}

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting benchmark: {len(MODELS)} model(s) × {CFG['epochs']} epochs"
         f"  [Thai folder dataset, {NUM_CLASSES} classes]", "bold"))
print(_c(f"{'═'*70}\n", "cyan"))

trained_models = {}
all_histories  = {}

for name, model_fn in MODELS.items():
    model, history = train_model(model_fn(), name)
    trained_models[name] = model
    all_histories[name]  = history

# ─────────────────────────────────────────────────────────────────────────────
#  10. FINAL TEST-SET EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
criterion_eval = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])
results = {}

for name, model in trained_models.items():
    test_loss, test_acc = evaluate(model, test_loader, criterion_eval)
    macro_f1            = compute_macro_f1(model, test_loader, NUM_CLASSES)
    results[name] = {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    sum(p.numel() for p in model.parameters()),
        "test_loss": round(float(test_loss), 4),
    }

print_comparison_table(results)

# ─────────────────────────────────────────────────────────────────────────────
#  11. PERSIST RESULTS
# ─────────────────────────────────────────────────────────────────────────────
results_path = os.path.join(CFG["results_dir"], "thai_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results   → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "thai_histories.json")
with open(histories_path, "w") as f:
    json.dump(
        {n: {k: [float(v) for v in vals] for k, vals in h.items()}
         for n, h in all_histories.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] Thai folder-dataset benchmark complete.\n", "green", "bold"))